In the modular approach I will create separate files for different model types. This one will be for beta_stack_model_sil_credo_score

# Define Library

In [1]:
# %% [markdown]
# # Jupyter Notebook Loading Header
#
# This is a custom loading header for Jupyter Notebooks in Visual Studio Code.
# It includes common imports and settings to get you started quickly.
# %% [markdown]
## Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.cloud import bigquery
from google.cloud import storage
import os
import tempfile
import time
from datetime import datetime
import uuid
import joblib
import uuid
from sklearn.metrics import roc_auc_score
from datetime import datetime, timedelta
import gcsfs
import duckdb as dd
import pickle
import joblib
from typing import Union
import io
path = r'C:\Users\Dwaipayan\AppData\Roaming\gcloud\legacy_credentials\dchakroborti@tonikbank.com\adc.json'
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = path
client = bigquery.Client(project='prj-prod-dataplatform')
os.environ["GOOGLE_CLOUD_PROJECT"] = "prj-prod-dataplatform"

# %% [markdown]
## Configure Settings
# Set options or configurations as needed
pd.set_option('display.max_columns', None)
pd.set_option("Display.max_rows", 100)

# Function

## calculate_gini

In [2]:
def calculate_gini(pd_scores, bad_indicators):
    """
    Calculate Gini coefficient from scores and binary indicators
    
    Parameters:
    pd_scores: array-like of scores/probabilities
    bad_indicators: array-like of binary outcomes (0/1)
    
    Returns:
    float: Gini coefficient
    """
    # Convert inputs to numpy arrays and ensure they're numeric
    pd_scores = np.array(pd_scores, dtype=float)
    bad_indicators = np.array(bad_indicators, dtype=int)
    
    # Check for valid input data
    if len(pd_scores) == 0 or len(bad_indicators) == 0:
        return np.nan
    
    # Check if we have both good and bad cases (needed for ROC AUC)
    if len(np.unique(bad_indicators)) < 2:
        return np.nan
    
    # Calculate AUC using sklearn
    try:
        auc = roc_auc_score(bad_indicators, pd_scores)
        # Calculate Gini from AUC
        gini = 2 * auc - 1
        return gini
    except ValueError:
        return np.nan

## calculate_periodic_gini_prod_ver_trench_dimfact

In [10]:
import pandas as pd
import numpy as np
from datetime import timedelta
from itertools import combinations

def calculate_gini(scores, labels):
    """
    Calculate Gini coefficient with proper handling of edge cases
    
    Returns np.nan when:
    - Fewer than 2 observations
    - No positive labels (sum of labels = 0)
    """
    n = len(scores)
    if n < 2:
        return np.nan
    
    label_sum = np.sum(labels)
    
    # Handle case where no positive labels exist (all zeros)
    # This prevents division by zero warning
    if label_sum == 0:
        return np.nan
    
    sorted_indices = np.argsort(scores)
    sorted_labels = labels.iloc[sorted_indices].values
    cumsum_labels = np.cumsum(sorted_labels)
    
    gini = 1 - 2 * np.sum(cumsum_labels) / (n * label_sum)
    return gini


def calculate_periodic_gini_prod_ver_trench_dimfact(df, score_column, label_column, namecolumn, 
                                        data_selection_column=None,
                                        model_version_column=None, trench_column=None, 
                                        loan_type_column=None, loan_product_type_column=None,
                                        ostype_column=None,
                                        risk_segment_column=None,
                                        risk_segment_final_column=None,
                                        account_id_column=None):
    """
    Calculate periodic Gini coefficients and return Power BI-friendly long format
    with fact and dimension tables.
    
    Returns:
    - fact_table: Long format with one row per segment per period
    - dimension_table: Unique segment combinations for filtering
    
    Parameters:
    df: DataFrame with disbursement dates and score/label columns
    score_column: name of the score column
    label_column: name of the label column
    namecolumn: name for the bad rate label
    data_selection_column: (optional) name of column for data selection (Test/Train)
    model_version_column: (optional) name of column for model version
    trench_column: (optional) name of column for trench category
    loan_type_column: (optional) name of loan type column
    loan_product_type_column: (optional) name of loan product type column
    ostype_column: (optional) name of column for OS type
    account_id_column: (optional) name of column for distinct account IDs
    """
    # Input validation
    required_columns = ['disbursementdate', score_column, label_column]
    if not all(col in df.columns for col in required_columns):
        raise ValueError(f"Missing required columns. Need: {required_columns}")
    
    optional_columns = {
        'data_selection': data_selection_column,
        'model_version': model_version_column,
        'trench': trench_column,
        'loan_type': loan_type_column,
        'loan_product_type': loan_product_type_column,
        'ostype': ostype_column,
        'risk_segment': risk_segment_column,
        'risk_segment_final': risk_segment_final_column,
        'account_id': account_id_column
    }
    
    for col_name, col in optional_columns.items():
        if col and col not in df.columns:
            raise ValueError(f"{col_name.replace('_', ' ').title()} column '{col}' not found in dataframe")
    
    # Create a copy to avoid modifying original dataframe
    df = df.copy()
    
    # Ensure date is datetime type
    df['disbursementdate'] = pd.to_datetime(df['disbursementdate'])
    
    # Ensure score and label columns are numeric
    df[score_column] = pd.to_numeric(df[score_column], errors='coerce')
    df[label_column] = pd.to_numeric(df[label_column], errors='coerce')
    
    # Drop rows with invalid values
    df = df.dropna(subset=[score_column, label_column])
    
    # Define list of datasets to process
    datasets_to_process = [('Overall', df, {})]
    
    # Create list of available segment columns
    segment_columns = []
    if data_selection_column:
        segment_columns.append(('DataSelection', data_selection_column))
    if model_version_column:
        segment_columns.append(('ModelVersion', model_version_column))
    if trench_column:
        segment_columns.append(('Trench', trench_column))
    if loan_type_column:
        segment_columns.append(('LoanType', loan_type_column))
    if loan_product_type_column:
        segment_columns.append(('ProductType', loan_product_type_column))
    if ostype_column:
        segment_columns.append(('OSType', ostype_column))
    if risk_segment_column:
        segment_columns.append(('risk_segment', risk_segment_column))
    if risk_segment_final_column:
        segment_columns.append(('risk_segment_final', risk_segment_final_column))
    
    # Generate all possible combinations of segment columns
    for r in range(1, len(segment_columns) + 1):
        for combo in combinations(segment_columns, r):
            def generate_combinations(df, segment_columns, index=0, current_filter=None, current_name=''):
                if current_filter is None:
                    current_filter = {}
                
                if index >= len(segment_columns):
                    filtered_df = df
                    for col, val in current_filter.items():
                        filtered_df = filtered_df[filtered_df[col] == val]
                    
                    if len(filtered_df) > 0:
                        yield (current_name.strip('_'), filtered_df, current_filter.copy())
                    return
                
                seg_name, seg_col = segment_columns[index]
                for seg_value in sorted(df[seg_col].dropna().unique()):
                    new_filter = current_filter.copy()
                    new_filter[seg_col] = seg_value
                    new_name = current_name + f'{seg_name}_{seg_value}_'
                    
                    yield from generate_combinations(df, segment_columns, index + 1, new_filter, new_name)
            
            for combo_name, combo_df, combo_metadata in generate_combinations(df, list(combo)):
                datasets_to_process.append((combo_name, combo_df, combo_metadata))
    
    all_results = []
    
    # Process each dataset
    for dataset_name, dataset_df, metadata in datasets_to_process:
        # Calculate weekly Gini
        dataset_df_copy = dataset_df.copy()
        dataset_df_copy['week'] = dataset_df_copy['disbursementdate'].dt.to_period('W')
        weekly_gini = dataset_df_copy.groupby('week').apply(
            lambda x: calculate_gini(x[score_column], x[label_column])
            if len(x) >= 10 else np.nan
        ).reset_index(name='gini_value')
        weekly_gini['period'] = 'Week'
        weekly_gini['start_date'] = weekly_gini['week'].apply(lambda x: x.to_timestamp())
        weekly_gini['end_date'] = weekly_gini['start_date'] + timedelta(days=6)
        
        # Add distinct account count for weekly
        if account_id_column:
            weekly_account_counts = dataset_df_copy.groupby('week')[account_id_column].nunique().reset_index()
            weekly_account_counts.columns = ['week', 'distinct_accounts']
            weekly_gini = weekly_gini.merge(weekly_account_counts, on='week', how='left')
        else:
            weekly_gini['distinct_accounts'] = None
        
        weekly_gini = weekly_gini[['start_date', 'end_date', 'gini_value', 'period', 'distinct_accounts']]
        
        # Calculate monthly Gini
        dataset_df_copy = dataset_df.copy()
        dataset_df_copy['month'] = dataset_df_copy['disbursementdate'].dt.to_period('M')
        monthly_gini = dataset_df_copy.groupby('month').apply(
            lambda x: calculate_gini(x[score_column], x[label_column])
            if len(x) >= 20 else np.nan
        ).reset_index(name='gini_value')
        monthly_gini['period'] = 'Month'
        monthly_gini['start_date'] = monthly_gini['month'].apply(lambda x: x.to_timestamp())
        monthly_gini['end_date'] = monthly_gini['start_date'] + pd.DateOffset(months=1) - pd.Timedelta(days=1)
        
        # Add distinct account count for monthly
        if account_id_column:
            monthly_account_counts = dataset_df_copy.groupby('month')[account_id_column].nunique().reset_index()
            monthly_account_counts.columns = ['month', 'distinct_accounts']
            monthly_gini = monthly_gini.merge(monthly_account_counts, on='month', how='left')
        else:
            monthly_gini['distinct_accounts'] = None
        
        monthly_gini = monthly_gini[['start_date', 'end_date', 'gini_value', 'period', 'distinct_accounts']]
        
        # Combine results for this dataset
        gini_results = pd.concat([weekly_gini, monthly_gini], ignore_index=True)
        gini_results = gini_results.sort_values(by='start_date').reset_index(drop=True)
        
        # Add metadata columns
        gini_results['Model_Name'] = score_column
        gini_results['bad_rate'] = namecolumn
        gini_results['segment_type'] = dataset_name
        
        # Add individual segment components
        gini_results['data_selection'] = metadata.get(data_selection_column, None) if data_selection_column else None
        gini_results['model_version'] = metadata.get(model_version_column, None) if model_version_column else None
        gini_results['trench_category'] = metadata.get(trench_column, None) if trench_column else None
        gini_results['loan_type'] = metadata.get(loan_type_column, None) if loan_type_column else None
        gini_results['loan_product_type'] = metadata.get(loan_product_type_column, None) if loan_product_type_column else None
        gini_results['ostype'] = metadata.get(ostype_column, None) if ostype_column else None
        gini_results['risk_segment'] = metadata.get(risk_segment_column, None) if risk_segment_column else None
        gini_results['risk_segment_final'] = metadata.get(risk_segment_final_column, None) if risk_segment_final_column else None   
        
        all_results.append(gini_results)
    
    # Combine all results
    fact_table = pd.concat(all_results, ignore_index=True)
    
    # Create dimension table (unique segment combinations for filtering)
    dimension_table = fact_table[['Model_Name', 'bad_rate', 'segment_type', 'data_selection', 'model_version', 
                                   'trench_category', 'loan_type', 'loan_product_type', 'ostype', 'risk_segment', 'risk_segment_final']].drop_duplicates().reset_index(drop=True)
    dimension_table['segment_id'] = range(len(dimension_table))
    
    # Add segment_id to fact table
    fact_table = fact_table.merge(dimension_table[['segment_id', 'Model_Name', 'bad_rate', 'segment_type', 
                                                     'data_selection', 'model_version', 'trench_category', 'loan_type', 'loan_product_type', 'ostype']], 
                                  on=['Model_Name', 'bad_rate', 'segment_type', 'data_selection', 'model_version', 
                                      'trench_category', 'loan_type', 'loan_product_type', 'ostype'], 
                                  how='left')
    
    # Reorder columns in fact table
    fact_table = fact_table[['segment_id', 'start_date', 'end_date', 'period', 'gini_value', 'distinct_accounts',
                             'Model_Name', 'bad_rate', 'segment_type', 'data_selection', 'model_version', 'trench_category', 
                             'loan_type', 'loan_product_type', 'ostype', 'risk_segment', 'risk_segment_final']]
    
    # Reorder columns in dimension table
    dimension_table = dimension_table[['segment_id', 'Model_Name', 'bad_rate', 'segment_type', 'data_selection', 
                                        'model_version', 'trench_category', 'loan_type', 'loan_product_type', 'ostype', 'risk_segment', 'risk_segment_final']]
    
    return fact_table, dimension_table


# Usage:
# fact_table, dimension_table = calculate_periodic_gini_prod_ver_trench_dimfact(
#     df_concat, 
#     'Alpha_cic_sil_score', 
#     'deffpd0', 
#     'FPD0',
#     data_selection_column='Data_selection',
#     model_version_column='modelVersionId',
#     trench_column='trenchCategory',
#     loan_type_column='loan_type',
#     loan_product_type_column='loan_product_type',
#     ostype_column='osType',
#     account_id_column='digitalLoanAccountId'
# )
# 
# # In Power BI:
# # 1. Import fact_table and dimension_table
# # 2. Create relationship: dimension_table[segment_id] -> fact_table[segment_id]
# # 3. Use dimension table columns as filters (including ostype)
# # 4. Create DAX measures:
# #    - Gini Measure = AVERAGE(fact_table[gini_value])
# #    - Account Count = SUM(fact_table[distinct_accounts])
# # 5. Use start_date, end_date, period for time-based analysis

# update_tables

In [4]:

def update_tables(fact_table: pd.DataFrame, dimension_table: pd.DataFrame, model_name: str, product: str) -> tuple:
    """
    Updates fact_table and dimension_table:
    - Sets 'Model_display_name' to the given model_name
    - Replaces NaN values in specified columns with 'Overall'
    
    Returns:
        Updated fact_table and dimension_table as a tuple
    """
    # Columns where missing values should be replaced
    cols_to_replace = ['model_version', 'trench_category', 'loan_type', 'loan_product_type', 'ostype']
    
    # Update fact_table
    fact_table['Model_display_name'] = model_name
    fact_table['Product_Category'] = product
    fact_table[cols_to_replace] = fact_table[cols_to_replace].fillna('Overall')
    
    # Update dimension_table
    dimension_table['Model_display_name'] = model_name
    dimension_table['Product_Category'] = product
    dimension_table[cols_to_replace] = dimension_table[cols_to_replace].fillna('Overall')
    
    return fact_table, dimension_table


In [5]:
facttable_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.fact_table_gamma"
dimtable_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.dimension_table_gamma"

## Gamma Cash Model Trench 1

## Trench 1

### FPD0 

### Test 

In [ ]:

sq = """
with b1 as 
(select * from prj-prod-dataplatform.audit_balance.ml_model_run_details 
where modelDisplayName like 'gamma_stack_model_cash' 
qualify row_number() over(partition by customerId order by publish_time desc) = 1
),
b2 as 
(select b1.customerId, lmt.digitalLoanAccountId, lmt.disbursementDateTime, b1.crifApplicationId 
, b1.prediction, b1.errorDetail,
b1.deployedModelId,
b1.model,
b1.modelDisplayName,
b1.modelVersionId,
b1.calcFeature,
b1.start_time,
b1.end_time,
b1.subscription_name,
b1.message_id,
b1.publish_time,
b1.attributes,
b1.trenchCategory,
b1.deviceOs
from b1
left join `risk_credit_mis.loan_master_table` lmt
on lmt.customerId = cast(b1.customerId as numeric)
and date(lmt.disbursementDateTime) > date(b1.publish_time)
where lmt.disbursementDateTime is not null
),
modelname as 
(select 
mmrd.customerId,mmrd.digitalLoanAccountId,prediction,start_time,end_time,modelDisplayName,modelVersionId, 
    case when trenchCategory is null then (case when mt.ln_user_type='1_Repeat Applicant' then 'Trench 3'
    when mt.ln_user_type <>'1_Repeat Applicant' and DATE_DIFF(current_date(), mt.onb_tsa_onboarding_datetime, DAY) >30 then 'Trench 2'
    else 'Trench1' end) 
     when trenchCategory = '' then (case when mt.ln_user_type='1_Repeat Applicant' then 'Trench 3'
    when mt.ln_user_type <>'1_Repeat Applicant' and DATE_DIFF(current_date(), mt.onb_tsa_onboarding_datetime, DAY) >30 then 'Trench 2'
    else 'Trench 1' end) 
    else trenchCategory end  as trenchCategory,
    REPLACE(REPLACE(calcFeature, "'", '"'), "None", "null") AS calcFeature,
    deviceOs osType,                                                                                                                --- Added ostype
  from b2 mmrd
  left join prj-prod-dataplatform.risk_credit_mis.model_loan_score_mart mt on mt.digitalLoanAccountId = mmrd.digitalLoanAccountId
)
,
deliquency as
(select loanAccountNumber,
case when obs_min_inst_def0 >= 1 and min_inst_def0 = 1 then 1 else 0 end deffpd0,
case when obs_min_inst_def10 >=1 and min_inst_def10 =1 then 1 else 0 end deffpd10,
case when obs_min_inst_def30 >=1 and min_inst_def30 =1 then 1 else 0 end deffpd30,
case when obs_min_inst_def30 >=2 and min_inst_def30 in (1,2) then 1 else 0 end deffspd30,
case when obs_min_inst_def30 >=3 and min_inst_def30 in (1,2,3) then 1 else 0 end deffstpd30,
case when obs_min_inst_def0 >= 1 then 1 else 0 end flg_mature_fpd0,
case when obs_min_inst_def10 >=1 then 1 else 0 end flg_mature_fpd10,
case when obs_min_inst_def30 >=1 then 1 else 0 end flg_mature_fpd30,
case when obs_min_inst_def30 >=2 then 1 else 0 end flg_mature_fspd_30,
case when obs_min_inst_def30 >=3 then 1 else 0 end flg_mature_fstpd_30
from prj-prod-dataplatform.risk_credit_mis.loan_deliquency_data),
base as 
  (select distinct r.customerId,
  r.digitalLoanAccountId,
  loanmaster.loanAccountNumber,
  r.modelDisplayName,
  r.prediction gamma_stack_model_cash_score,
  coalesce(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime),  cast(r.start_time as datetime)) AS appln_submit_datetime,
  date(loanmaster.disbursementDateTime) disbursementdate,
  format_date('%Y-%m', coalesce(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime),  cast(r.start_time as datetime))) as Application_month, 
  'Prod' Data_selection,  
  deffpd0,
  flg_mature_fpd0,
  loanmaster.new_loan_type,
  modelVersionId, trenchCategory,
    case when loanmaster.loantype='BNPL' and store_type =1 then 'Appliance'
    when loanmaster.loantype='BNPL' and store_type =2 then 'Mobile' 
    when loanmaster.loantype='BNPL' and store_type =3 then 'Mall' 
    when loanmaster.loantype='BNPL' and store_type not in (1,2,3) then store_tagging
    else 'not applicable' end as loan_product_type,
    coalesce((case when lower(r.osType) like '%andro%' then 'android'
                  when lower(r.osType) like '%os%' then 'ios' else lower(r.osType) end), 
            (case when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%andro%' then 'android'
                  when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%os%' then 'ios'
                  when lower(loanmaster.deviceType) like '%andro%' then 'android'
                  else 'ios' end)
            ) as osType
  from modelname r
  left join risk_credit_mis.loan_master_table loanmaster  ON loanmaster.digitalLoanAccountId = r.digitalLoanAccountId
  inner join deliquency del on del.loanAccountNumber = loanmaster.loanAccountNumber
   left join(SELECT DISTINCT mer_refferal_code, mer_name mer_name,store_type,store_tagging FROM `dl_loans_db_raw.tdbk_merchant_refferal_mtb`
  left join worktable_datachampions.TARGET_SPLIT P on P.STORE_NAME = mer_name
 qualify row_number() over(partition by mer_refferal_code order by  created_dt desc)=1) sil_category on loanmaster.purpleKey=sil_category.mer_refferal_code
  where loanmaster.flagDisbursement = 1
  and loanmaster.disbursementDateTime is not null
  and r.prediction is not null
  and flg_mature_fpd0 = 1
  ),
  segmentdata AS (
SELECT loan.customerid , loan.digitalLoanAccountId, trench_category.trenchCategory,
loan.offer_id,
CASE
WHEN COALESCE(trench1_seg.risk_segment) IS NULL
THEN 'Unsegmented'
ELSE COALESCE(trench1_seg.risk_segment)
END AS risk_segment,
appVersion,flagApproval,tsa_onboarding_time,
IF(applicationStatus  in ('COMPLETED','ACTIVATED','APPROVED') , 'Loan Approved', 'Loan Not Approved') AS loan_application_status,
--if(disbursementDateTime is not null, 'Loan Disbursed', 'Loan Not Approved') loan_application_status
DATE(decision_date) AS application_date  
from
(select distinct digitalLoanAccountId, customerId,applicationStatus,disbursementDateTime,date(decision_date) decision_date, appVersion,flagApproval,tsa_onboarding_time,offer_id from `risk_credit_mis.loan_master_table`
  where date(decision_date) >=date('2025-11-10') and new_loan_type='Quick'
--QUALIFY ROW_NUMBER() OVER(PARTITION BY customerId ORDER BY decision_date desc)=1
) loan
left join (SELECT digitalLoanAccountId,case when trenchCategory='Trench 1' then 'Trench-1'  when trenchCategory='Trench 2' then 'Trench-2'
when trenchCategory='Trench 3' then 'Trench-3' end as trenchCategory
,publish_time FROM `audit_balance.ml_model_run_details`
where modelDisplayName='gamma_stack_model_cash'
qualify row_number() over(partition by digitalLoanAccountId order by publish_time desc)=1
) trench_category on trench_category.digitalLoanAccountId=loan.digitalLoanAccountId
LEFT JOIN (
SELECT
cust_id,risk_segment,created_date,created_by,offer_id FROM `dl_loans_db_raw.tdbk_loan_offers_trx`
WHERE offer_type='SEGMENTED_ACL'
--AND created_by='GCP-API-CALL'
--QUALIFY ROW_NUMBER() OVER(PARTITION BY cust_id ORDER BY created_date desc)=1
) trench1_seg ON trench1_seg.offer_id=loan.offer_id

),
b3 as 
(select 
  b.customerId,  
  b.digitalLoanAccountId,
  b.loanAccountNumber,
  b.modelDisplayName,
  b.gamma_stack_model_cash_score,
  b.appln_submit_datetime,
  b.disbursementdate,
  b.Application_month,
  b.Data_selection,
  b.deffpd0,
  b.flg_mature_fpd0,
  b.new_loan_type,
  b.modelVersionId,
  b.trenchCategory,
  b.loan_product_type,
  b.osType,
  sd.offer_id,
  sd.risk_segment,
  frs.risk_segment_final
from base b
left join segmentdata sd on sd.digitalLoanAccountId = b.digitalLoanAccountId
LEFT JOIN (
  select digitalLoanAccountid, risk_segment_final from prj-prod-dataplatform.dl_loans_db_raw.tdbk_loan_poi3_response where risk_segment_final is not null
qualify row_number() over(partition by digitalLoanAccountid order by created_dt desc) = 1
          ) frs on frs.digitalLoanAccountId = b.digitalLoanAccountId
qualify row_number() over(partition by b.digitalLoanAccountId, b.modelVersionId order by b.appln_submit_datetime) = 1
)
select * from b3 where lower(new_loan_type) not like '%sil%'
  ;

"""
dfd = client.query(sq).to_dataframe()
# dfd = dfd.drop_duplicates(keep='first')
print(f"The shape of the dataframe downloaded is:\t {dfd.shape}")
dfd.head()



The shape of the dataframe downloaded is:	 (3818, 19)


,customerId,digitalLoanAccountId,loanAccountNumber,modelDisplayName,gamma_stack_model_cash_score,appln_submit_datetime,disbursementdate,Application_month,Data_selection,deffpd0,flg_mature_fpd0,new_loan_type,modelVersionId,trenchCategory,loan_product_type,osType,offer_id,risk_segment,risk_segment_final
0,3805966,5f088ba4-7720-4865-9ec0-9f4e4d7588a0,60838059660016,gamma_stack_model_cash,0.395949466411459,2025-11-12 05:42:41,2025-11-12,2025-11,Prod,0,1,Quick,v1,Trench 1,not applicable,ios,981152,Segment2,None
1,3809915,1788e0e9-e42a-4cf1-8113-f5b5df0f1c6c,60838099150011,gamma_stack_model_cash,0.21497827644647646,2025-11-14 15:18:11,2025-11-15,2025-11,Prod,0,1,Quick,v1,Trench 1,not applicable,android,1414675,Segment4,None
2,3805992,7046357e-3cc1-4d19-a948-084cc170503d,60838059920018,gamma_stack_model_cash,0.13310559212754985,2025-11-12 07:13:56,2025-11-12,2025-11,Prod,0,1,Quick,v1,Trench 1,not applicable,android,981174,Segment3,None
3,3805539,2b4ce052-9177-45fe-9273-e4940c336a3c,60838055390011,gamma_stack_model_cash,0.4181553645751821,2025-11-12 08:26:51,2025-11-12,2025-11,Prod,0,1,Quick,v1,Trench 1,not applicable,ios,980781,Segment2,None
4,3805834,6006ff4f-9cdb-4120-8b29-468158e89b77,60838058340015,gamma_stack_model_cash,0.4923946224891471,2025-11-11 21:07:12,2025-11-12,2025-11,Prod,0,1,Quick,v1,Trench 1,not applicable,ios,981041,Segment2,None


In [7]:
df1 = dfd.copy()

### Train 

In [ ]:
# sq = """
# with modelname as 
#   (  SELECT
#     mmrd.customerId,mmrd.digitalLoanAccountId,prediction,start_time,end_time,modelDisplayName,modelVersionId, 
#   case when trenchCategory is null then (case when mt.ln_user_type='1_Repeat Applicant' then 'Trench 3'
#     when mt.ln_user_type <>'1_Repeat Applicant' and DATE_DIFF(current_date(), mt.onb_tsa_onboarding_datetime, DAY) >30 then 'Trench 2'
#     else 'Trench1' end) 
#      when trenchCategory = '' then (case when mt.ln_user_type='1_Repeat Applicant' then 'Trench 3'
#     when mt.ln_user_type <>'1_Repeat Applicant' and DATE_DIFF(current_date(), mt.onb_tsa_onboarding_datetime, DAY) >30 then 'Trench 2'
#     else 'Trench 1' end) 
#     else trenchCategory end  as trenchCategory,
#     REPLACE(REPLACE(calcFeature, "'", '"'), "None", "null") AS calcFeature,
#     Data_selection, deviceOs osType,
#     FROM prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116 mmrd
#   left join prj-prod-dataplatform.risk_credit_mis.model_loan_score_mart mt on mt.digitalLoanAccountId = mmrd.digitalLoanAccountId
#   WHERE modelDisplayName in ('Alpha-Cash-Stack-Model', 'alpha_stack_model_cash')
#   -- and modelVersionId = 'v1'
#       ),
#   deliquency as
# (select loanAccountNumber,
# case when obs_min_inst_def0 >= 1 and min_inst_def0 = 1 then 1 else 0 end deffpd0,
# case when obs_min_inst_def10 >=1 and min_inst_def10 =1 then 1 else 0 end deffpd10,
# case when obs_min_inst_def30 >=1 and min_inst_def30 =1 then 1 else 0 end deffpd30,
# case when obs_min_inst_def30 >=2 and min_inst_def30 in (1,2) then 1 else 0 end deffspd30,
# case when obs_min_inst_def30 >=3 and min_inst_def30 in (1,2,3) then 1 else 0 end deffstpd30,
# case when obs_min_inst_def0 >= 1 then 1 else 0 end flg_mature_fpd0,
# case when obs_min_inst_def10 >=1 then 1 else 0 end flg_mature_fpd10,
# case when obs_min_inst_def30 >=1 then 1 else 0 end flg_mature_fpd30,
# case when obs_min_inst_def30 >=2 then 1 else 0 end flg_mature_fspd_30,
# case when obs_min_inst_def30 >=3 then 1 else 0 end flg_mature_fstpd_30
# from prj-prod-dataplatform.risk_credit_mis.loan_deliquency_data),
# base as 
#   (select distinct r.customerId,
#   r.digitalLoanAccountId,
#   loanmaster.loanAccountNumber,
#   r.modelDisplayName,
#   coalesce(cast(JSON_VALUE(SAFE.PARSE_JSON(CAST(calcFeature AS STRING)), '$.credo_score')AS FLOAT64), 
#   cast(JSON_VALUE(SAFE.PARSE_JSON(CAST(calcFeature AS STRING)), '$.c_credo_score')AS FLOAT64)) AS credo_score,
#   calcFeature,
#   coalesce(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime),  cast(r.start_time as datetime)) AS appln_submit_datetime,
#   date(loanmaster.disbursementDateTime) disbursementdate,
#   format_date('%Y-%m', coalesce(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime),  cast(r.start_time as datetime))) as Application_month, 
#   Data_selection,  
#   deffpd0,
#   flg_mature_fpd0,
#   loanmaster.new_loan_type,
#   modelVersionId, trenchCategory,
#     case when loanmaster.loantype='BNPL' and store_type =1 then 'Appliance'
#     when loanmaster.loantype='BNPL' and store_type =2 then 'Mobile' 
#     when loanmaster.loantype='BNPL' and store_type =3 then 'Mall' 
#     when loanmaster.loantype='BNPL' and store_type not in (1,2,3) then store_tagging
#     else 'not applicable' end as loan_product_type,
#        coalesce((case when lower(r.osType) like '%andro%' then 'android'
#                   when lower(r.osType) like '%os%' then 'ios' else lower(r.osType) end), 
#             (case when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%andro%' then 'android'
#                   when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%os%' then 'ios'
#                   when lower(loanmaster.deviceType) like '%andro%' then 'android'
#                   else 'ios' end)
#             ) as osType
#   from modelname r
#   left join risk_credit_mis.loan_master_table loanmaster  ON loanmaster.digitalLoanAccountId = r.digitalLoanAccountId
#   left join deliquency del on del.loanAccountNumber = loanmaster.loanAccountNumber
#    left join(SELECT DISTINCT mer_refferal_code, mer_name mer_name,store_type,store_tagging FROM `dl_loans_db_raw.tdbk_merchant_refferal_mtb`
#   left join worktable_datachampions.TARGET_SPLIT P on P.STORE_NAME = mer_name
#  qualify row_number() over(partition by mer_refferal_code order by  created_dt desc)=1) sil_category on loanmaster.purpleKey=sil_category.mer_refferal_code
#   where 
#   loanmaster.flagDisbursement = 1
#   and loanmaster.disbursementDateTime is not null
#   and flg_mature_fpd0 = 1
#   )
#   select *
#   from base
#   where credo_score is not null
#  qualify row_number() over(partition by digitalLoanAccountId, modelVersionId order by appln_submit_datetime) = 1
#    ;
#   """
# dfd = client.query(sq).to_dataframe()
# # dfd = dfd.drop_duplicates(keep='first')
# print(f"The shape of the dataframe downloaded is:\t {dfd.shape}")
# dfd.head()

In [ ]:
# df2 = dfd.copy()

### Concat 

In [8]:
# # df_concat = pd.concat([df1, df2], ignore_index=True)
# # print(f"The shape of the concatenated dataframe is:\t {df_concat.shape}")

# # df_concat = (df_concat
# #              .sort_values(by=['digitalLoanAccountId', 'Data_selection'],
# #                           key=lambda s: s.map({'Train': 0, 'Test': 1}))
# #              .drop_duplicates(subset=['digitalLoanAccountId'], keep='first')
# #              .reset_index(drop=True))
# # print(f"The shape of the concatenated dataframe is:\t {df_concat.shape}")
# # df_concat.head()


# # 1) Get all IDs present in Train
# train_ids = set(df2['digitalLoanAccountId'])

# # 2) Keep only Test rows whose ID is NOT in Train
# df1_no_dupes = df1[~df1['digitalLoanAccountId'].isin(train_ids)]

# # 3) Concatenate
# df_concat = pd.concat([df1_no_dupes, df2], ignore_index=True)

# print(f"The shape of the concatenated dataframe is:\t {df_concat.shape}")
# df_concat.head()

df_concat = df1.copy()


In [9]:
# # df_concat = df1.copy()

df_concat['gamma_stack_model_cash_score'] = pd.to_numeric(df_concat['gamma_stack_model_cash_score'], errors='coerce')

In [11]:
df_concat['risk_segment'] = df_concat['risk_segment'].fillna('NA')
df_concat['risk_segment_final'] = df_concat['risk_segment_final'].fillna('NA')

In [12]:
fact_table, dimension_table = calculate_periodic_gini_prod_ver_trench_dimfact(
    df_concat, 
    'gamma_stack_model_cash_score', 
    'deffpd0', 
    'FPD0',
    data_selection_column='Data_selection',
    model_version_column='modelVersionId',
    trench_column='trenchCategory',
    loan_type_column='new_loan_type',
    loan_product_type_column='loan_product_type',
    ostype_column='osType',  
    risk_segment_column='risk_segment',
    risk_segment_final_column='risk_segment_final',
    account_id_column='digitalLoanAccountId'
)

In [13]:
fact_table, dimension_table = update_tables(fact_table, dimension_table, model_name='gamma_stack_model_cash', product='CASH')
print(f"The shape of the fact table is:\t {fact_table.shape}")
print(f"The shape of the dimension table is:\t {dimension_table.shape}")

The shape of the fact table is:	 (15200, 19)
The shape of the dimension table is:	 (1440, 14)


In [ ]:
# Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.fact_table3"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(fact_table, facttable_id, job_config=job_config)
job.result()  # Wait for the job to complete

C:\Users\Dwaipayan\AppData\Roaming\Python\Python312\site-packages\google\cloud\bigquery\_pandas_helpers.py:483: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=616eb147-5165-431d-93a9-5a196fab92d1>

In [ ]:
# Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.dimension_table3"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dimension_table, dimtable_id, job_config=job_config)
job.result()  # Wait for the job to complete

C:\Users\Dwaipayan\AppData\Roaming\Python\Python312\site-packages\google\cloud\bigquery\_pandas_helpers.py:483: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=fa3b08ad-9e99-4dd3-a145-91e24796e3bc>

### FPD10 

### Test 

In [ ]:

sq = """
with b1 as 
(select * from prj-prod-dataplatform.audit_balance.ml_model_run_details 
where modelDisplayName like 'gamma_stack_model_cash' 
qualify row_number() over(partition by customerId order by publish_time desc) = 1
),
b2 as 
(select b1.customerId, lmt.digitalLoanAccountId, lmt.disbursementDateTime, b1.crifApplicationId 
, b1.prediction, b1.errorDetail,
b1.deployedModelId,
b1.model,
b1.modelDisplayName,
b1.modelVersionId,
b1.calcFeature,
b1.start_time,
b1.end_time,
b1.subscription_name,
b1.message_id,
b1.publish_time,
b1.attributes,
b1.trenchCategory,
b1.deviceOs
from b1
left join `risk_credit_mis.loan_master_table` lmt
on lmt.customerId = cast(b1.customerId as numeric)
and date(lmt.disbursementDateTime) > date(b1.publish_time)
where lmt.disbursementDateTime is not null
),
modelname as 
(select 
mmrd.customerId,mmrd.digitalLoanAccountId,prediction,start_time,end_time,modelDisplayName,modelVersionId, 
    case when trenchCategory is null then (case when mt.ln_user_type='1_Repeat Applicant' then 'Trench 3'
    when mt.ln_user_type <>'1_Repeat Applicant' and DATE_DIFF(current_date(), mt.onb_tsa_onboarding_datetime, DAY) >30 then 'Trench 2'
    else 'Trench1' end) 
     when trenchCategory = '' then (case when mt.ln_user_type='1_Repeat Applicant' then 'Trench 3'
    when mt.ln_user_type <>'1_Repeat Applicant' and DATE_DIFF(current_date(), mt.onb_tsa_onboarding_datetime, DAY) >30 then 'Trench 2'
    else 'Trench 1' end) 
    else trenchCategory end  as trenchCategory,
    REPLACE(REPLACE(calcFeature, "'", '"'), "None", "null") AS calcFeature,
    deviceOs osType,                                                                                                                --- Added ostype
  from b2 mmrd
  left join prj-prod-dataplatform.risk_credit_mis.model_loan_score_mart mt on mt.digitalLoanAccountId = mmrd.digitalLoanAccountId
)
,
deliquency as
(select loanAccountNumber,
case when obs_min_inst_def0 >= 1 and min_inst_def0 = 1 then 1 else 0 end deffpd0,
case when obs_min_inst_def10 >=1 and min_inst_def10 =1 then 1 else 0 end deffpd10,
case when obs_min_inst_def30 >=1 and min_inst_def30 =1 then 1 else 0 end deffpd30,
case when obs_min_inst_def30 >=2 and min_inst_def30 in (1,2) then 1 else 0 end deffspd30,
case when obs_min_inst_def30 >=3 and min_inst_def30 in (1,2,3) then 1 else 0 end deffstpd30,
case when obs_min_inst_def0 >= 1 then 1 else 0 end flg_mature_fpd0,
case when obs_min_inst_def10 >=1 then 1 else 0 end flg_mature_fpd10,
case when obs_min_inst_def30 >=1 then 1 else 0 end flg_mature_fpd30,
case when obs_min_inst_def30 >=2 then 1 else 0 end flg_mature_fspd_30,
case when obs_min_inst_def30 >=3 then 1 else 0 end flg_mature_fstpd_30
from prj-prod-dataplatform.risk_credit_mis.loan_deliquency_data),
base as 
  (select distinct r.customerId,
  r.digitalLoanAccountId,
  loanmaster.loanAccountNumber,
  r.modelDisplayName,
  r.prediction gamma_stack_model_cash_score,
  coalesce(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime),  cast(r.start_time as datetime)) AS appln_submit_datetime,
  date(loanmaster.disbursementDateTime) disbursementdate,
  format_date('%Y-%m', coalesce(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime),  cast(r.start_time as datetime))) as Application_month, 
  'Prod' Data_selection,  
  del.deffpd10,
  del.flg_mature_fpd10,
  loanmaster.new_loan_type,
  modelVersionId, trenchCategory,
    case when loanmaster.loantype='BNPL' and store_type =1 then 'Appliance'
    when loanmaster.loantype='BNPL' and store_type =2 then 'Mobile' 
    when loanmaster.loantype='BNPL' and store_type =3 then 'Mall' 
    when loanmaster.loantype='BNPL' and store_type not in (1,2,3) then store_tagging
    else 'not applicable' end as loan_product_type,
    coalesce((case when lower(r.osType) like '%andro%' then 'android'
                  when lower(r.osType) like '%os%' then 'ios' else lower(r.osType) end), 
            (case when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%andro%' then 'android'
                  when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%os%' then 'ios'
                  when lower(loanmaster.deviceType) like '%andro%' then 'android'
                  else 'ios' end)
            ) as osType
  from modelname r
  left join risk_credit_mis.loan_master_table loanmaster  ON loanmaster.digitalLoanAccountId = r.digitalLoanAccountId
  inner join deliquency del on del.loanAccountNumber = loanmaster.loanAccountNumber
   left join(SELECT DISTINCT mer_refferal_code, mer_name mer_name,store_type,store_tagging FROM `dl_loans_db_raw.tdbk_merchant_refferal_mtb`
  left join worktable_datachampions.TARGET_SPLIT P on P.STORE_NAME = mer_name
 qualify row_number() over(partition by mer_refferal_code order by  created_dt desc)=1) sil_category on loanmaster.purpleKey=sil_category.mer_refferal_code
  where loanmaster.flagDisbursement = 1
  and loanmaster.disbursementDateTime is not null
  and r.prediction is not null
  and del.flg_mature_fpd10 = 1
  ),
  segmentdata AS (
SELECT loan.customerid , loan.digitalLoanAccountId, trench_category.trenchCategory,
loan.offer_id,
CASE
WHEN COALESCE(trench1_seg.risk_segment) IS NULL
THEN 'Unsegmented'
ELSE COALESCE(trench1_seg.risk_segment)
END AS risk_segment,
appVersion,flagApproval,tsa_onboarding_time,
IF(applicationStatus  in ('COMPLETED','ACTIVATED','APPROVED') , 'Loan Approved', 'Loan Not Approved') AS loan_application_status,
--if(disbursementDateTime is not null, 'Loan Disbursed', 'Loan Not Approved') loan_application_status
DATE(decision_date) AS application_date  
from
(select distinct digitalLoanAccountId, customerId,applicationStatus,disbursementDateTime,date(decision_date) decision_date, appVersion,flagApproval,tsa_onboarding_time,offer_id from `risk_credit_mis.loan_master_table`
  where date(decision_date) >=date('2025-11-10') and new_loan_type='Quick'
--QUALIFY ROW_NUMBER() OVER(PARTITION BY customerId ORDER BY decision_date desc)=1
) loan
left join (SELECT digitalLoanAccountId,case when trenchCategory='Trench 1' then 'Trench-1'  when trenchCategory='Trench 2' then 'Trench-2'
when trenchCategory='Trench 3' then 'Trench-3' end as trenchCategory
,publish_time FROM `audit_balance.ml_model_run_details`
where modelDisplayName='gamma_stack_model_cash'
qualify row_number() over(partition by digitalLoanAccountId order by publish_time desc)=1
) trench_category on trench_category.digitalLoanAccountId=loan.digitalLoanAccountId
LEFT JOIN (
SELECT
cust_id,risk_segment,created_date,created_by,offer_id FROM `dl_loans_db_raw.tdbk_loan_offers_trx`
WHERE offer_type='SEGMENTED_ACL'
--AND created_by='GCP-API-CALL'
--QUALIFY ROW_NUMBER() OVER(PARTITION BY cust_id ORDER BY created_date desc)=1
) trench1_seg ON trench1_seg.offer_id=loan.offer_id

),
b3 as 
(select 
  b.customerId,  
  b.digitalLoanAccountId,
  b.loanAccountNumber,
  b.modelDisplayName,
  b.gamma_stack_model_cash_score,
  b.appln_submit_datetime,
  b.disbursementdate,
  b.Application_month,
  b.Data_selection,
  b.deffpd10,
  b.flg_mature_fpd10,
  b.new_loan_type,
  b.modelVersionId,
  b.trenchCategory,
  b.loan_product_type,
  b.osType,
  sd.offer_id,
  sd.risk_segment,
  frs.risk_segment_final
from base b
left join segmentdata sd on sd.digitalLoanAccountId = b.digitalLoanAccountId
LEFT JOIN (
  select digitalLoanAccountid, risk_segment_final from prj-prod-dataplatform.dl_loans_db_raw.tdbk_loan_poi3_response where risk_segment_final is not null
qualify row_number() over(partition by digitalLoanAccountid order by created_dt desc) = 1
          ) frs on frs.digitalLoanAccountId = b.digitalLoanAccountId
qualify row_number() over(partition by b.digitalLoanAccountId, b.modelVersionId order by b.appln_submit_datetime) = 1
)
select * from b3 where lower(new_loan_type) not like '%sil%'
  ;




"""
dfd = client.query(sq).to_dataframe()
# dfd = dfd.drop_duplicates(keep='first')
print(f"The shape of the dataframe downloaded is:\t {dfd.shape}")
dfd.head()



The shape of the dataframe downloaded is:	 (2858, 19)


,customerId,digitalLoanAccountId,loanAccountNumber,modelDisplayName,gamma_stack_model_cash_score,appln_submit_datetime,disbursementdate,Application_month,Data_selection,deffpd10,flg_mature_fpd10,new_loan_type,modelVersionId,trenchCategory,loan_product_type,osType,offer_id,risk_segment,risk_segment_final
0,3810799,19470c11-feb8-4169-bffd-5285e292af73,60838107990014,gamma_stack_model_cash,0.04620008769017148,2025-11-15 06:00:42,2025-11-15,2025-11,Prod,1,1,Quick,v1,Trench 1,not applicable,android,1752021,Segment1,None
1,3896661,bba53dec-e1fa-45db-91dc-fcfc3588b0e8,60838966610011,gamma_stack_model_cash,0.1336068484427431,2025-12-15 16:17:17,2025-12-31,2025-12,Prod,0,1,Quick,v1,Trench 1,not applicable,android,2994482,Segment3,None
2,3805003,d67f8be0-dbc6-41b9-9dca-a91120b664df,60838050030014,gamma_stack_model_cash,0.3367484717370137,2025-11-12 17:47:15,2025-11-12,2025-11,Prod,0,1,Quick,v1,Trench 1,not applicable,ios,942182,Segment2,None
3,3825791,5e2d4136-2ed4-4fb9-a3a2-e98f11bfbee0,60838257910011,gamma_stack_model_cash,0.15652293065783943,2025-12-13 17:46:22,2025-12-28,2025-12,Prod,1,1,Quick,v1,Trench 1,not applicable,android,2220886,Segment4,None
4,3935885,4824639f-f765-4546-a82d-d071787f4e67,60839358850017,gamma_stack_model_cash,0.5026425353918945,2025-12-25 23:37:17,2025-12-26,2025-12,Prod,0,1,Quick,v1,Trench 1,not applicable,android,3883290,Segment2,None


In [17]:
df1 = dfd.copy()

### Train 

In [18]:
# sq = """
# with modelname as 
#   (  SELECT
#     mmrd.customerId,mmrd.digitalLoanAccountId,prediction,start_time,end_time,modelDisplayName,modelVersionId, 
#   case when trenchCategory is null then (case when mt.ln_user_type='1_Repeat Applicant' then 'Trench 3'
#     when mt.ln_user_type <>'1_Repeat Applicant' and DATE_DIFF(current_date(), mt.onb_tsa_onboarding_datetime, DAY) >30 then 'Trench 2'
#     else 'Trench1' end) 
#      when trenchCategory = '' then (case when mt.ln_user_type='1_Repeat Applicant' then 'Trench 3'
#     when mt.ln_user_type <>'1_Repeat Applicant' and DATE_DIFF(current_date(), mt.onb_tsa_onboarding_datetime, DAY) >30 then 'Trench 2'
#     else 'Trench 1' end) 
#     else trenchCategory end  as trenchCategory,
#     REPLACE(REPLACE(calcFeature, "'", '"'), "None", "null") AS calcFeature,
#     Data_selection, deviceOs osType,
#     FROM prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116 mmrd
#   left join prj-prod-dataplatform.risk_credit_mis.model_loan_score_mart mt on mt.digitalLoanAccountId = mmrd.digitalLoanAccountId
#   WHERE modelDisplayName in ('Alpha-Cash-Stack-Model', 'alpha_stack_model_cash')
#   -- and modelVersionId = 'v1'
#       ),
#   deliquency as
# (select loanAccountNumber,
# case when obs_min_inst_def0 >= 1 and min_inst_def0 = 1 then 1 else 0 end deffpd0,
# case when obs_min_inst_def10 >=1 and min_inst_def10 =1 then 1 else 0 end deffpd10,
# case when obs_min_inst_def30 >=1 and min_inst_def30 =1 then 1 else 0 end deffpd30,
# case when obs_min_inst_def30 >=2 and min_inst_def30 in (1,2) then 1 else 0 end deffspd30,
# case when obs_min_inst_def30 >=3 and min_inst_def30 in (1,2,3) then 1 else 0 end deffstpd30,
# case when obs_min_inst_def0 >= 1 then 1 else 0 end flg_mature_fpd0,
# case when obs_min_inst_def10 >=1 then 1 else 0 end flg_mature_fpd10,
# case when obs_min_inst_def30 >=1 then 1 else 0 end flg_mature_fpd30,
# case when obs_min_inst_def30 >=2 then 1 else 0 end flg_mature_fspd_30,
# case when obs_min_inst_def30 >=3 then 1 else 0 end flg_mature_fstpd_30
# from prj-prod-dataplatform.risk_credit_mis.loan_deliquency_data),
# base as 
#   (select distinct r.customerId,
#   r.digitalLoanAccountId,
#   loanmaster.loanAccountNumber,
#   r.modelDisplayName,
#   coalesce(cast(JSON_VALUE(SAFE.PARSE_JSON(CAST(calcFeature AS STRING)), '$.credo_score')AS FLOAT64), 
#   cast(JSON_VALUE(SAFE.PARSE_JSON(CAST(calcFeature AS STRING)), '$.c_credo_score')AS FLOAT64)) AS credo_score,
#   calcFeature,
#   coalesce(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime),  cast(r.start_time as datetime)) AS appln_submit_datetime,
#   date(loanmaster.disbursementDateTime) disbursementdate,
#   format_date('%Y-%m', coalesce(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime),  cast(r.start_time as datetime))) as Application_month, 
#   Data_selection,  
#   deffpd0,
#   flg_mature_fpd0,
#   loanmaster.new_loan_type,
#   modelVersionId, trenchCategory,
#     case when loanmaster.loantype='BNPL' and store_type =1 then 'Appliance'
#     when loanmaster.loantype='BNPL' and store_type =2 then 'Mobile' 
#     when loanmaster.loantype='BNPL' and store_type =3 then 'Mall' 
#     when loanmaster.loantype='BNPL' and store_type not in (1,2,3) then store_tagging
#     else 'not applicable' end as loan_product_type,
#        coalesce((case when lower(r.osType) like '%andro%' then 'android'
#                   when lower(r.osType) like '%os%' then 'ios' else lower(r.osType) end), 
#             (case when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%andro%' then 'android'
#                   when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%os%' then 'ios'
#                   when lower(loanmaster.deviceType) like '%andro%' then 'android'
#                   else 'ios' end)
#             ) as osType
#   from modelname r
#   left join risk_credit_mis.loan_master_table loanmaster  ON loanmaster.digitalLoanAccountId = r.digitalLoanAccountId
#   left join deliquency del on del.loanAccountNumber = loanmaster.loanAccountNumber
#    left join(SELECT DISTINCT mer_refferal_code, mer_name mer_name,store_type,store_tagging FROM `dl_loans_db_raw.tdbk_merchant_refferal_mtb`
#   left join worktable_datachampions.TARGET_SPLIT P on P.STORE_NAME = mer_name
#  qualify row_number() over(partition by mer_refferal_code order by  created_dt desc)=1) sil_category on loanmaster.purpleKey=sil_category.mer_refferal_code
#   where 
#   loanmaster.flagDisbursement = 1
#   and loanmaster.disbursementDateTime is not null
#   and flg_mature_fpd0 = 1
#   )
#   select *
#   from base
#   where credo_score is not null
#  qualify row_number() over(partition by digitalLoanAccountId, modelVersionId order by appln_submit_datetime) = 1
#    ;
#   """
# dfd = client.query(sq).to_dataframe()
# # dfd = dfd.drop_duplicates(keep='first')
# print(f"The shape of the dataframe downloaded is:\t {dfd.shape}")
# dfd.head()

In [19]:
# df2 = dfd.copy()

### Concat 

In [20]:
# # df_concat = pd.concat([df1, df2], ignore_index=True)
# # print(f"The shape of the concatenated dataframe is:\t {df_concat.shape}")

# # df_concat = (df_concat
# #              .sort_values(by=['digitalLoanAccountId', 'Data_selection'],
# #                           key=lambda s: s.map({'Train': 0, 'Test': 1}))
# #              .drop_duplicates(subset=['digitalLoanAccountId'], keep='first')
# #              .reset_index(drop=True))
# # print(f"The shape of the concatenated dataframe is:\t {df_concat.shape}")
# # df_concat.head()


# # 1) Get all IDs present in Train
# train_ids = set(df2['digitalLoanAccountId'])

# # 2) Keep only Test rows whose ID is NOT in Train
# df1_no_dupes = df1[~df1['digitalLoanAccountId'].isin(train_ids)]

# # 3) Concatenate
# df_concat = pd.concat([df1_no_dupes, df2], ignore_index=True)

# print(f"The shape of the concatenated dataframe is:\t {df_concat.shape}")
# df_concat.head()

df_concat = df1.copy()


In [21]:
# # df_concat = df1.copy()

df_concat['gamma_stack_model_cash_score'] = pd.to_numeric(df_concat['gamma_stack_model_cash_score'], errors='coerce')

In [22]:
df_concat['risk_segment'] = df_concat['risk_segment'].fillna('NA')
df_concat['risk_segment_final'] = df_concat['risk_segment_final'].fillna('NA')

In [23]:
fact_table, dimension_table = calculate_periodic_gini_prod_ver_trench_dimfact(
    df_concat, 
    'gamma_stack_model_cash_score', 
    'deffpd10', 
    'FPD10',
    data_selection_column='Data_selection',
    model_version_column='modelVersionId',
    trench_column='trenchCategory',
    loan_type_column='new_loan_type',
    loan_product_type_column='loan_product_type',
    ostype_column='osType',  
    risk_segment_column='risk_segment',
    risk_segment_final_column='risk_segment_final',
    account_id_column='digitalLoanAccountId'
)

In [24]:
fact_table, dimension_table = update_tables(fact_table, dimension_table, model_name='gamma_stack_model_cash', product='CASH')
print(f"The shape of the fact table is:\t {fact_table.shape}")
print(f"The shape of the dimension table is:\t {dimension_table.shape}")

The shape of the fact table is:	 (12800, 19)
The shape of the dimension table is:	 (1440, 14)


In [25]:
# Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.fact_table3"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(fact_table, facttable_id, job_config=job_config)
job.result()  # Wait for the job to complete

C:\Users\Dwaipayan\AppData\Roaming\Python\Python312\site-packages\google\cloud\bigquery\_pandas_helpers.py:483: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=a6f3c309-742b-43d4-a547-46bd01eeeabf>

In [26]:
# Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.dimension_table3"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dimension_table, dimtable_id, job_config=job_config)
job.result()  # Wait for the job to complete

C:\Users\Dwaipayan\AppData\Roaming\Python\Python312\site-packages\google\cloud\bigquery\_pandas_helpers.py:483: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=2afe2c73-a72a-4098-8fe8-ab3dae54203f>

### FPD30 

### Test 

In [ ]:

sq = """
with b1 as 
(select * from prj-prod-dataplatform.audit_balance.ml_model_run_details 
where modelDisplayName like 'gamma_stack_model_cash' 
qualify row_number() over(partition by customerId order by publish_time desc) = 1
),
b2 as 
(select b1.customerId, lmt.digitalLoanAccountId, lmt.disbursementDateTime, b1.crifApplicationId 
, b1.prediction, b1.errorDetail,
b1.deployedModelId,
b1.model,
b1.modelDisplayName,
b1.modelVersionId,
b1.calcFeature,
b1.start_time,
b1.end_time,
b1.subscription_name,
b1.message_id,
b1.publish_time,
b1.attributes,
b1.trenchCategory,
b1.deviceOs
from b1
left join `risk_credit_mis.loan_master_table` lmt
on lmt.customerId = cast(b1.customerId as numeric)
and date(lmt.disbursementDateTime) > date(b1.publish_time)
where lmt.disbursementDateTime is not null
),
modelname as 
(select 
mmrd.customerId,mmrd.digitalLoanAccountId,prediction,start_time,end_time,modelDisplayName,modelVersionId, 
    case when trenchCategory is null then (case when mt.ln_user_type='1_Repeat Applicant' then 'Trench 3'
    when mt.ln_user_type <>'1_Repeat Applicant' and DATE_DIFF(current_date(), mt.onb_tsa_onboarding_datetime, DAY) >30 then 'Trench 2'
    else 'Trench1' end) 
     when trenchCategory = '' then (case when mt.ln_user_type='1_Repeat Applicant' then 'Trench 3'
    when mt.ln_user_type <>'1_Repeat Applicant' and DATE_DIFF(current_date(), mt.onb_tsa_onboarding_datetime, DAY) >30 then 'Trench 2'
    else 'Trench 1' end) 
    else trenchCategory end  as trenchCategory,
    REPLACE(REPLACE(calcFeature, "'", '"'), "None", "null") AS calcFeature,
    deviceOs osType,                                                                                                                --- Added ostype
  from b2 mmrd
  left join prj-prod-dataplatform.risk_credit_mis.model_loan_score_mart mt on mt.digitalLoanAccountId = mmrd.digitalLoanAccountId
)
,
deliquency as
(select loanAccountNumber,
case when obs_min_inst_def0 >= 1 and min_inst_def0 = 1 then 1 else 0 end deffpd0,
case when obs_min_inst_def10 >=1 and min_inst_def10 =1 then 1 else 0 end deffpd10,
case when obs_min_inst_def30 >=1 and min_inst_def30 =1 then 1 else 0 end deffpd30,
case when obs_min_inst_def30 >=2 and min_inst_def30 in (1,2) then 1 else 0 end deffspd30,
case when obs_min_inst_def30 >=3 and min_inst_def30 in (1,2,3) then 1 else 0 end deffstpd30,
case when obs_min_inst_def0 >= 1 then 1 else 0 end flg_mature_fpd0,
case when obs_min_inst_def10 >=1 then 1 else 0 end flg_mature_fpd10,
case when obs_min_inst_def30 >=1 then 1 else 0 end flg_mature_fpd30,
case when obs_min_inst_def30 >=2 then 1 else 0 end flg_mature_fspd_30,
case when obs_min_inst_def30 >=3 then 1 else 0 end flg_mature_fstpd_30
from prj-prod-dataplatform.risk_credit_mis.loan_deliquency_data),
base as 
  (select distinct r.customerId,
  r.digitalLoanAccountId,
  loanmaster.loanAccountNumber,
  r.modelDisplayName,
  r.prediction gamma_stack_model_cash_score,
  coalesce(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime),  cast(r.start_time as datetime)) AS appln_submit_datetime,
  date(loanmaster.disbursementDateTime) disbursementdate,
  format_date('%Y-%m', coalesce(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime),  cast(r.start_time as datetime))) as Application_month, 
  'Prod' Data_selection,  
  del.deffpd30,
  del.flg_mature_fpd30,
  loanmaster.new_loan_type,
  modelVersionId, trenchCategory,
    case when loanmaster.loantype='BNPL' and store_type =1 then 'Appliance'
    when loanmaster.loantype='BNPL' and store_type =2 then 'Mobile' 
    when loanmaster.loantype='BNPL' and store_type =3 then 'Mall' 
    when loanmaster.loantype='BNPL' and store_type not in (1,2,3) then store_tagging
    else 'not applicable' end as loan_product_type,
    coalesce((case when lower(r.osType) like '%andro%' then 'android'
                  when lower(r.osType) like '%os%' then 'ios' else lower(r.osType) end), 
            (case when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%andro%' then 'android'
                  when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%os%' then 'ios'
                  when lower(loanmaster.deviceType) like '%andro%' then 'android'
                  else 'ios' end)
            ) as osType
  from modelname r
  left join risk_credit_mis.loan_master_table loanmaster  ON loanmaster.digitalLoanAccountId = r.digitalLoanAccountId
  inner join deliquency del on del.loanAccountNumber = loanmaster.loanAccountNumber
   left join(SELECT DISTINCT mer_refferal_code, mer_name mer_name,store_type,store_tagging FROM `dl_loans_db_raw.tdbk_merchant_refferal_mtb`
  left join worktable_datachampions.TARGET_SPLIT P on P.STORE_NAME = mer_name
 qualify row_number() over(partition by mer_refferal_code order by  created_dt desc)=1) sil_category on loanmaster.purpleKey=sil_category.mer_refferal_code
  where loanmaster.flagDisbursement = 1
  and loanmaster.disbursementDateTime is not null
  and r.prediction is not null
  and del.flg_mature_fpd30 = 1
  ),
  segmentdata AS (
SELECT loan.customerid , loan.digitalLoanAccountId, trench_category.trenchCategory,
loan.offer_id,
CASE
WHEN COALESCE(trench1_seg.risk_segment) IS NULL
THEN 'Unsegmented'
ELSE COALESCE(trench1_seg.risk_segment)
END AS risk_segment,
appVersion,flagApproval,tsa_onboarding_time,
IF(applicationStatus  in ('COMPLETED','ACTIVATED','APPROVED') , 'Loan Approved', 'Loan Not Approved') AS loan_application_status,
--if(disbursementDateTime is not null, 'Loan Disbursed', 'Loan Not Approved') loan_application_status
DATE(decision_date) AS application_date  
from
(select distinct digitalLoanAccountId, customerId,applicationStatus,disbursementDateTime,date(decision_date) decision_date, appVersion,flagApproval,tsa_onboarding_time,offer_id from `risk_credit_mis.loan_master_table`
  where date(decision_date) >=date('2025-11-10') and new_loan_type='Quick'
--QUALIFY ROW_NUMBER() OVER(PARTITION BY customerId ORDER BY decision_date desc)=1
) loan
left join (SELECT digitalLoanAccountId,case when trenchCategory='Trench 1' then 'Trench-1'  when trenchCategory='Trench 2' then 'Trench-2'
when trenchCategory='Trench 3' then 'Trench-3' end as trenchCategory
,publish_time FROM `audit_balance.ml_model_run_details`
where modelDisplayName='gamma_stack_model_cash'
qualify row_number() over(partition by digitalLoanAccountId order by publish_time desc)=1
) trench_category on trench_category.digitalLoanAccountId=loan.digitalLoanAccountId
LEFT JOIN (
SELECT
cust_id,risk_segment,created_date,created_by,offer_id FROM `dl_loans_db_raw.tdbk_loan_offers_trx`
WHERE offer_type='SEGMENTED_ACL'
--AND created_by='GCP-API-CALL'
--QUALIFY ROW_NUMBER() OVER(PARTITION BY cust_id ORDER BY created_date desc)=1
) trench1_seg ON trench1_seg.offer_id=loan.offer_id
),
b3 as 
(select 
  b.customerId,  
  b.digitalLoanAccountId,
  b.loanAccountNumber,
  b.modelDisplayName,
  b.gamma_stack_model_cash_score,
  b.appln_submit_datetime,
  b.disbursementdate,
  b.Application_month,
  b.Data_selection,
  b.deffpd30,
  b.flg_mature_fpd30,
  b.new_loan_type,
  b.modelVersionId,
  b.trenchCategory,
  b.loan_product_type,
  b.osType,
  sd.offer_id,
  sd.risk_segment,
  frs.risk_segment_final
from base b
left join segmentdata sd on sd.digitalLoanAccountId = b.digitalLoanAccountId
LEFT JOIN (
  select digitalLoanAccountid, risk_segment_final from prj-prod-dataplatform.dl_loans_db_raw.tdbk_loan_poi3_response where risk_segment_final is not null
qualify row_number() over(partition by digitalLoanAccountid order by created_dt desc) = 1
          ) frs on frs.digitalLoanAccountId = b.digitalLoanAccountId
qualify row_number() over(partition by b.digitalLoanAccountId, b.modelVersionId order by b.appln_submit_datetime) = 1
)
select * from b3 where lower(new_loan_type) not like '%sil%'
  ;
"""
dfd = client.query(sq).to_dataframe()
# dfd = dfd.drop_duplicates(keep='first')
print(f"The shape of the dataframe downloaded is:\t {dfd.shape}")
dfd.head()



The shape of the dataframe downloaded is:	 (1099, 19)


,customerId,digitalLoanAccountId,loanAccountNumber,modelDisplayName,gamma_stack_model_cash_score,appln_submit_datetime,disbursementdate,Application_month,Data_selection,deffpd30,flg_mature_fpd30,new_loan_type,modelVersionId,trenchCategory,loan_product_type,osType,offer_id,risk_segment,risk_segment_final
0,3860433,1bdeb385-814a-4fde-a416-f0b12988585a,60838604330012,gamma_stack_model_cash,0.128427778740863,2025-12-04 20:03:34,2025-12-05,2025-12,Prod,0,1,Quick,v1,Trench 1,not applicable,android,2458889,Segment3,None
1,3841590,1c8ca856-7988-43b2-bb84-030d8cd4233c,60838415900018,gamma_stack_model_cash,0.27036772274587223,2025-11-28 06:15:19,2025-12-01,2025-11,Prod,0,1,Quick,v1,Trench 1,not applicable,android,2299631,Segment4,None
2,3816044,2d9ec6d5-c2b1-4db9-a7ab-1f86d4f636ca,60838160440012,gamma_stack_model_cash,0.06738460168688715,2025-11-17 14:37:40,2025-11-19,2025-11,Prod,1,1,Quick,v1,Trench 1,not applicable,android,2207746,Segment1,None
3,3823965,300870cc-ae2c-463e-b4d5-3ae07c9f8a33,60838239650017,gamma_stack_model_cash,0.45417581067402885,2025-11-21 01:13:34,2025-11-21,2025-11,Prod,0,1,Quick,v1,Trench 1,not applicable,ios,2217722,Segment2,None
4,3826791,34badeb9-1e95-4e56-a0a6-74a9383f1ff8,60838267910015,gamma_stack_model_cash,0.5390120915850531,2025-11-27 08:36:04,2025-11-27,2025-11,Prod,0,1,Quick,v1,Trench 1,not applicable,ios,None,Unsegmented,None


In [28]:
df1 = dfd.copy()

### Train 

In [29]:
# sq = """
# with modelname as 
#   (  SELECT
#     mmrd.customerId,mmrd.digitalLoanAccountId,prediction,start_time,end_time,modelDisplayName,modelVersionId, 
#   case when trenchCategory is null then (case when mt.ln_user_type='1_Repeat Applicant' then 'Trench 3'
#     when mt.ln_user_type <>'1_Repeat Applicant' and DATE_DIFF(current_date(), mt.onb_tsa_onboarding_datetime, DAY) >30 then 'Trench 2'
#     else 'Trench1' end) 
#      when trenchCategory = '' then (case when mt.ln_user_type='1_Repeat Applicant' then 'Trench 3'
#     when mt.ln_user_type <>'1_Repeat Applicant' and DATE_DIFF(current_date(), mt.onb_tsa_onboarding_datetime, DAY) >30 then 'Trench 2'
#     else 'Trench 1' end) 
#     else trenchCategory end  as trenchCategory,
#     REPLACE(REPLACE(calcFeature, "'", '"'), "None", "null") AS calcFeature,
#     Data_selection, deviceOs osType,
#     FROM prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116 mmrd
#   left join prj-prod-dataplatform.risk_credit_mis.model_loan_score_mart mt on mt.digitalLoanAccountId = mmrd.digitalLoanAccountId
#   WHERE modelDisplayName in ('Alpha-Cash-Stack-Model', 'alpha_stack_model_cash')
#   -- and modelVersionId = 'v1'
#       ),
#   deliquency as
# (select loanAccountNumber,
# case when obs_min_inst_def0 >= 1 and min_inst_def0 = 1 then 1 else 0 end deffpd0,
# case when obs_min_inst_def10 >=1 and min_inst_def10 =1 then 1 else 0 end deffpd10,
# case when obs_min_inst_def30 >=1 and min_inst_def30 =1 then 1 else 0 end deffpd30,
# case when obs_min_inst_def30 >=2 and min_inst_def30 in (1,2) then 1 else 0 end deffspd30,
# case when obs_min_inst_def30 >=3 and min_inst_def30 in (1,2,3) then 1 else 0 end deffstpd30,
# case when obs_min_inst_def0 >= 1 then 1 else 0 end flg_mature_fpd0,
# case when obs_min_inst_def10 >=1 then 1 else 0 end flg_mature_fpd10,
# case when obs_min_inst_def30 >=1 then 1 else 0 end flg_mature_fpd30,
# case when obs_min_inst_def30 >=2 then 1 else 0 end flg_mature_fspd_30,
# case when obs_min_inst_def30 >=3 then 1 else 0 end flg_mature_fstpd_30
# from prj-prod-dataplatform.risk_credit_mis.loan_deliquency_data),
# base as 
#   (select distinct r.customerId,
#   r.digitalLoanAccountId,
#   loanmaster.loanAccountNumber,
#   r.modelDisplayName,
#   coalesce(cast(JSON_VALUE(SAFE.PARSE_JSON(CAST(calcFeature AS STRING)), '$.credo_score')AS FLOAT64), 
#   cast(JSON_VALUE(SAFE.PARSE_JSON(CAST(calcFeature AS STRING)), '$.c_credo_score')AS FLOAT64)) AS credo_score,
#   calcFeature,
#   coalesce(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime),  cast(r.start_time as datetime)) AS appln_submit_datetime,
#   date(loanmaster.disbursementDateTime) disbursementdate,
#   format_date('%Y-%m', coalesce(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime),  cast(r.start_time as datetime))) as Application_month, 
#   Data_selection,  
#   deffpd0,
#   flg_mature_fpd0,
#   loanmaster.new_loan_type,
#   modelVersionId, trenchCategory,
#     case when loanmaster.loantype='BNPL' and store_type =1 then 'Appliance'
#     when loanmaster.loantype='BNPL' and store_type =2 then 'Mobile' 
#     when loanmaster.loantype='BNPL' and store_type =3 then 'Mall' 
#     when loanmaster.loantype='BNPL' and store_type not in (1,2,3) then store_tagging
#     else 'not applicable' end as loan_product_type,
#        coalesce((case when lower(r.osType) like '%andro%' then 'android'
#                   when lower(r.osType) like '%os%' then 'ios' else lower(r.osType) end), 
#             (case when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%andro%' then 'android'
#                   when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%os%' then 'ios'
#                   when lower(loanmaster.deviceType) like '%andro%' then 'android'
#                   else 'ios' end)
#             ) as osType
#   from modelname r
#   left join risk_credit_mis.loan_master_table loanmaster  ON loanmaster.digitalLoanAccountId = r.digitalLoanAccountId
#   left join deliquency del on del.loanAccountNumber = loanmaster.loanAccountNumber
#    left join(SELECT DISTINCT mer_refferal_code, mer_name mer_name,store_type,store_tagging FROM `dl_loans_db_raw.tdbk_merchant_refferal_mtb`
#   left join worktable_datachampions.TARGET_SPLIT P on P.STORE_NAME = mer_name
#  qualify row_number() over(partition by mer_refferal_code order by  created_dt desc)=1) sil_category on loanmaster.purpleKey=sil_category.mer_refferal_code
#   where 
#   loanmaster.flagDisbursement = 1
#   and loanmaster.disbursementDateTime is not null
#   and flg_mature_fpd0 = 1
#   )
#   select *
#   from base
#   where credo_score is not null
#  qualify row_number() over(partition by digitalLoanAccountId, modelVersionId order by appln_submit_datetime) = 1
#    ;
#   """
# dfd = client.query(sq).to_dataframe()
# # dfd = dfd.drop_duplicates(keep='first')
# print(f"The shape of the dataframe downloaded is:\t {dfd.shape}")
# dfd.head()

In [30]:
# df2 = dfd.copy()

### Concat 

In [31]:
# # df_concat = pd.concat([df1, df2], ignore_index=True)
# # print(f"The shape of the concatenated dataframe is:\t {df_concat.shape}")

# # df_concat = (df_concat
# #              .sort_values(by=['digitalLoanAccountId', 'Data_selection'],
# #                           key=lambda s: s.map({'Train': 0, 'Test': 1}))
# #              .drop_duplicates(subset=['digitalLoanAccountId'], keep='first')
# #              .reset_index(drop=True))
# # print(f"The shape of the concatenated dataframe is:\t {df_concat.shape}")
# # df_concat.head()


# # 1) Get all IDs present in Train
# train_ids = set(df2['digitalLoanAccountId'])

# # 2) Keep only Test rows whose ID is NOT in Train
# df1_no_dupes = df1[~df1['digitalLoanAccountId'].isin(train_ids)]

# # 3) Concatenate
# df_concat = pd.concat([df1_no_dupes, df2], ignore_index=True)

# print(f"The shape of the concatenated dataframe is:\t {df_concat.shape}")
# df_concat.head()

df_concat = df1.copy()


In [32]:
# # df_concat = df1.copy()

df_concat['gamma_stack_model_cash_score'] = pd.to_numeric(df_concat['gamma_stack_model_cash_score'], errors='coerce')

In [33]:
df_concat['risk_segment'] = df_concat['risk_segment'].fillna('NA')
df_concat['risk_segment_final'] = df_concat['risk_segment_final'].fillna('NA')

In [34]:
fact_table, dimension_table = calculate_periodic_gini_prod_ver_trench_dimfact(
    df_concat, 
    'gamma_stack_model_cash_score', 
    'deffpd30', 
    'FPD30',
    data_selection_column='Data_selection',
    model_version_column='modelVersionId',
    trench_column='trenchCategory',
    loan_type_column='new_loan_type',
    loan_product_type_column='loan_product_type',
    ostype_column='osType',  
    risk_segment_column='risk_segment',
    risk_segment_final_column='risk_segment_final',
    account_id_column='digitalLoanAccountId'
)

In [35]:
fact_table, dimension_table = update_tables(fact_table, dimension_table, model_name='gamma_stack_model_cash', product='CASH')
print(f"The shape of the fact table is:\t {fact_table.shape}")
print(f"The shape of the dimension table is:\t {dimension_table.shape}")

The shape of the fact table is:	 (7744, 19)
The shape of the dimension table is:	 (1376, 14)


In [36]:
# Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.fact_table3"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(fact_table, facttable_id, job_config=job_config)
job.result()  # Wait for the job to complete

C:\Users\Dwaipayan\AppData\Roaming\Python\Python312\site-packages\google\cloud\bigquery\_pandas_helpers.py:483: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=484fbd75-e8c9-4ae6-98e8-a776f955e588>

In [37]:
# Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.dimension_table3"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dimension_table, dimtable_id, job_config=job_config)
job.result()  # Wait for the job to complete

C:\Users\Dwaipayan\AppData\Roaming\Python\Python312\site-packages\google\cloud\bigquery\_pandas_helpers.py:483: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=d9f4b2e0-c7b0-4788-9dfa-842b0134357e>

### FSPD30 

### Test 

In [ ]:

sq = """
with b1 as 
(select * from prj-prod-dataplatform.audit_balance.ml_model_run_details 
where modelDisplayName like 'gamma_stack_model_cash' 
qualify row_number() over(partition by customerId order by publish_time desc) = 1
),
b2 as 
(select b1.customerId, lmt.digitalLoanAccountId, lmt.disbursementDateTime, b1.crifApplicationId 
, b1.prediction, b1.errorDetail,
b1.deployedModelId,
b1.model,
b1.modelDisplayName,
b1.modelVersionId,
b1.calcFeature,
b1.start_time,
b1.end_time,
b1.subscription_name,
b1.message_id,
b1.publish_time,
b1.attributes,
b1.trenchCategory,
b1.deviceOs
from b1
left join `risk_credit_mis.loan_master_table` lmt
on lmt.customerId = cast(b1.customerId as numeric)
and date(lmt.disbursementDateTime) > date(b1.publish_time)
where lmt.disbursementDateTime is not null
),
modelname as 
(select 
mmrd.customerId,mmrd.digitalLoanAccountId,prediction,start_time,end_time,modelDisplayName,modelVersionId, 
    case when trenchCategory is null then (case when mt.ln_user_type='1_Repeat Applicant' then 'Trench 3'
    when mt.ln_user_type <>'1_Repeat Applicant' and DATE_DIFF(current_date(), mt.onb_tsa_onboarding_datetime, DAY) >30 then 'Trench 2'
    else 'Trench1' end) 
     when trenchCategory = '' then (case when mt.ln_user_type='1_Repeat Applicant' then 'Trench 3'
    when mt.ln_user_type <>'1_Repeat Applicant' and DATE_DIFF(current_date(), mt.onb_tsa_onboarding_datetime, DAY) >30 then 'Trench 2'
    else 'Trench 1' end) 
    else trenchCategory end  as trenchCategory,
    REPLACE(REPLACE(calcFeature, "'", '"'), "None", "null") AS calcFeature,
    deviceOs osType,                                                                                                                --- Added ostype
  from b2 mmrd
  left join prj-prod-dataplatform.risk_credit_mis.model_loan_score_mart mt on mt.digitalLoanAccountId = mmrd.digitalLoanAccountId
)
,
deliquency as
(select loanAccountNumber,
case when obs_min_inst_def0 >= 1 and min_inst_def0 = 1 then 1 else 0 end deffpd0,
case when obs_min_inst_def10 >=1 and min_inst_def10 =1 then 1 else 0 end deffpd10,
case when obs_min_inst_def30 >=1 and min_inst_def30 =1 then 1 else 0 end deffpd30,
case when obs_min_inst_def30 >=2 and min_inst_def30 in (1,2) then 1 else 0 end deffspd30,
case when obs_min_inst_def30 >=3 and min_inst_def30 in (1,2,3) then 1 else 0 end deffstpd30,
case when obs_min_inst_def0 >= 1 then 1 else 0 end flg_mature_fpd0,
case when obs_min_inst_def10 >=1 then 1 else 0 end flg_mature_fpd10,
case when obs_min_inst_def30 >=1 then 1 else 0 end flg_mature_fpd30,
case when obs_min_inst_def30 >=2 then 1 else 0 end flg_mature_fspd_30,
case when obs_min_inst_def30 >=3 then 1 else 0 end flg_mature_fstpd_30
from prj-prod-dataplatform.risk_credit_mis.loan_deliquency_data),
base as 
  (select distinct r.customerId,
  r.digitalLoanAccountId,
  loanmaster.loanAccountNumber,
  r.modelDisplayName,
  r.prediction gamma_stack_model_cash_score,
  coalesce(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime),  cast(r.start_time as datetime)) AS appln_submit_datetime,
  date(loanmaster.disbursementDateTime) disbursementdate,
  format_date('%Y-%m', coalesce(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime),  cast(r.start_time as datetime))) as Application_month, 
  'Prod' Data_selection,  
  del.deffspd30,
  del.flg_mature_fspd_30,
  loanmaster.new_loan_type,
  modelVersionId, trenchCategory,
    case when loanmaster.loantype='BNPL' and store_type =1 then 'Appliance'
    when loanmaster.loantype='BNPL' and store_type =2 then 'Mobile' 
    when loanmaster.loantype='BNPL' and store_type =3 then 'Mall' 
    when loanmaster.loantype='BNPL' and store_type not in (1,2,3) then store_tagging
    else 'not applicable' end as loan_product_type,
    coalesce((case when lower(r.osType) like '%andro%' then 'android'
                  when lower(r.osType) like '%os%' then 'ios' else lower(r.osType) end), 
            (case when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%andro%' then 'android'
                  when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%os%' then 'ios'
                  when lower(loanmaster.deviceType) like '%andro%' then 'android'
                  else 'ios' end)
            ) as osType
  from modelname r
  left join risk_credit_mis.loan_master_table loanmaster  ON loanmaster.digitalLoanAccountId = r.digitalLoanAccountId
  inner join deliquency del on del.loanAccountNumber = loanmaster.loanAccountNumber
   left join(SELECT DISTINCT mer_refferal_code, mer_name mer_name,store_type,store_tagging FROM `dl_loans_db_raw.tdbk_merchant_refferal_mtb`
  left join worktable_datachampions.TARGET_SPLIT P on P.STORE_NAME = mer_name
 qualify row_number() over(partition by mer_refferal_code order by  created_dt desc)=1) sil_category on loanmaster.purpleKey=sil_category.mer_refferal_code
  where loanmaster.flagDisbursement = 1
  and loanmaster.disbursementDateTime is not null
  and r.prediction is not null
  and del.flg_mature_fspd_30 = 1
  ),
  segmentdata AS (
SELECT loan.customerid , loan.digitalLoanAccountId, trench_category.trenchCategory,
loan.offer_id,
CASE
WHEN COALESCE(trench1_seg.risk_segment) IS NULL
THEN 'Unsegmented'
ELSE COALESCE(trench1_seg.risk_segment)
END AS risk_segment,
appVersion,flagApproval,tsa_onboarding_time,
IF(applicationStatus  in ('COMPLETED','ACTIVATED','APPROVED') , 'Loan Approved', 'Loan Not Approved') AS loan_application_status,
--if(disbursementDateTime is not null, 'Loan Disbursed', 'Loan Not Approved') loan_application_status
DATE(decision_date) AS application_date  
from
(select distinct digitalLoanAccountId, customerId,applicationStatus,disbursementDateTime,date(decision_date) decision_date, appVersion,flagApproval,tsa_onboarding_time,offer_id from `risk_credit_mis.loan_master_table`
  where date(decision_date) >=date('2025-11-10') and new_loan_type='Quick'
--QUALIFY ROW_NUMBER() OVER(PARTITION BY customerId ORDER BY decision_date desc)=1
) loan
left join (SELECT digitalLoanAccountId,case when trenchCategory='Trench 1' then 'Trench-1'  when trenchCategory='Trench 2' then 'Trench-2'
when trenchCategory='Trench 3' then 'Trench-3' end as trenchCategory
,publish_time FROM `audit_balance.ml_model_run_details`
where modelDisplayName='gamma_stack_model_cash'
qualify row_number() over(partition by digitalLoanAccountId order by publish_time desc)=1
) trench_category on trench_category.digitalLoanAccountId=loan.digitalLoanAccountId
LEFT JOIN (
SELECT
cust_id,risk_segment,created_date,created_by,offer_id FROM `dl_loans_db_raw.tdbk_loan_offers_trx`
WHERE offer_type='SEGMENTED_ACL'
--AND created_by='GCP-API-CALL'
--QUALIFY ROW_NUMBER() OVER(PARTITION BY cust_id ORDER BY created_date desc)=1
) trench1_seg ON trench1_seg.offer_id=loan.offer_id

),
b3 as 
(select 
  b.customerId,  
  b.digitalLoanAccountId,
  b.loanAccountNumber,
  b.modelDisplayName,
  b.gamma_stack_model_cash_score,
  b.appln_submit_datetime,
  b.disbursementdate,
  b.Application_month,
  b.Data_selection,
  b.deffspd30,
  b.flg_mature_fspd_30 flg_mature_fspd30,
  b.new_loan_type,
  b.modelVersionId,
  b.trenchCategory,
  b.loan_product_type,
  b.osType,
  sd.offer_id,
  sd.risk_segment,
  frs.risk_segment_final
from base b
left join segmentdata sd on sd.digitalLoanAccountId = b.digitalLoanAccountId
LEFT JOIN (
  select digitalLoanAccountid, risk_segment_final from prj-prod-dataplatform.dl_loans_db_raw.tdbk_loan_poi3_response where risk_segment_final is not null
qualify row_number() over(partition by digitalLoanAccountid order by created_dt desc) = 1
          ) frs on frs.digitalLoanAccountId = b.digitalLoanAccountId
qualify row_number() over(partition by b.digitalLoanAccountId, b.modelVersionId order by b.appln_submit_datetime) = 1
)
select * from b3 where lower(new_loan_type) not like '%sil%'
;
"""
dfd = client.query(sq).to_dataframe()
# dfd = dfd.drop_duplicates(keep='first')
print(f"The shape of the dataframe downloaded is:\t {dfd.shape}")
dfd.head()



The shape of the dataframe downloaded is:	 (0, 19)


,customerId,digitalLoanAccountId,loanAccountNumber,modelDisplayName,gamma_stack_model_cash_score,appln_submit_datetime,disbursementdate,Application_month,Data_selection,deffspd30,flg_mature_fspd30,new_loan_type,modelVersionId,trenchCategory,loan_product_type,osType,offer_id,risk_segment,risk_segment_final


In [39]:
df1 = dfd.copy()

### Train 

In [ ]:
# sq = """
# with modelname as 
#   (  SELECT
#     mmrd.customerId,mmrd.digitalLoanAccountId,prediction,start_time,end_time,modelDisplayName,modelVersionId, 
#   case when trenchCategory is null then (case when mt.ln_user_type='1_Repeat Applicant' then 'Trench 3'
#     when mt.ln_user_type <>'1_Repeat Applicant' and DATE_DIFF(current_date(), mt.onb_tsa_onboarding_datetime, DAY) >30 then 'Trench 2'
#     else 'Trench1' end) 
#      when trenchCategory = '' then (case when mt.ln_user_type='1_Repeat Applicant' then 'Trench 3'
#     when mt.ln_user_type <>'1_Repeat Applicant' and DATE_DIFF(current_date(), mt.onb_tsa_onboarding_datetime, DAY) >30 then 'Trench 2'
#     else 'Trench 1' end) 
#     else trenchCategory end  as trenchCategory,
#     REPLACE(REPLACE(calcFeature, "'", '"'), "None", "null") AS calcFeature,
#     Data_selection, deviceOs osType,
#     FROM prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116 mmrd
#   left join prj-prod-dataplatform.risk_credit_mis.model_loan_score_mart mt on mt.digitalLoanAccountId = mmrd.digitalLoanAccountId
#   WHERE modelDisplayName in ('Alpha-Cash-Stack-Model', 'alpha_stack_model_cash')
#   -- and modelVersionId = 'v1'
#       ),
#   deliquency as
# (select loanAccountNumber,
# case when obs_min_inst_def0 >= 1 and min_inst_def0 = 1 then 1 else 0 end deffpd0,
# case when obs_min_inst_def10 >=1 and min_inst_def10 =1 then 1 else 0 end deffpd10,
# case when obs_min_inst_def30 >=1 and min_inst_def30 =1 then 1 else 0 end deffpd30,
# case when obs_min_inst_def30 >=2 and min_inst_def30 in (1,2) then 1 else 0 end deffspd30,
# case when obs_min_inst_def30 >=3 and min_inst_def30 in (1,2,3) then 1 else 0 end deffstpd30,
# case when obs_min_inst_def0 >= 1 then 1 else 0 end flg_mature_fpd0,
# case when obs_min_inst_def10 >=1 then 1 else 0 end flg_mature_fpd10,
# case when obs_min_inst_def30 >=1 then 1 else 0 end flg_mature_fpd30,
# case when obs_min_inst_def30 >=2 then 1 else 0 end flg_mature_fspd_30,
# case when obs_min_inst_def30 >=3 then 1 else 0 end flg_mature_fstpd_30
# from prj-prod-dataplatform.risk_credit_mis.loan_deliquency_data),
# base as 
#   (select distinct r.customerId,
#   r.digitalLoanAccountId,
#   loanmaster.loanAccountNumber,
#   r.modelDisplayName,
#   coalesce(cast(JSON_VALUE(SAFE.PARSE_JSON(CAST(calcFeature AS STRING)), '$.credo_score')AS FLOAT64), 
#   cast(JSON_VALUE(SAFE.PARSE_JSON(CAST(calcFeature AS STRING)), '$.c_credo_score')AS FLOAT64)) AS credo_score,
#   calcFeature,
#   coalesce(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime),  cast(r.start_time as datetime)) AS appln_submit_datetime,
#   date(loanmaster.disbursementDateTime) disbursementdate,
#   format_date('%Y-%m', coalesce(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime),  cast(r.start_time as datetime))) as Application_month, 
#   Data_selection,  
#   deffpd0,
#   flg_mature_fpd0,
#   loanmaster.new_loan_type,
#   modelVersionId, trenchCategory,
#     case when loanmaster.loantype='BNPL' and store_type =1 then 'Appliance'
#     when loanmaster.loantype='BNPL' and store_type =2 then 'Mobile' 
#     when loanmaster.loantype='BNPL' and store_type =3 then 'Mall' 
#     when loanmaster.loantype='BNPL' and store_type not in (1,2,3) then store_tagging
#     else 'not applicable' end as loan_product_type,
#        coalesce((case when lower(r.osType) like '%andro%' then 'android'
#                   when lower(r.osType) like '%os%' then 'ios' else lower(r.osType) end), 
#             (case when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%andro%' then 'android'
#                   when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%os%' then 'ios'
#                   when lower(loanmaster.deviceType) like '%andro%' then 'android'
#                   else 'ios' end)
#             ) as osType
#   from modelname r
#   left join risk_credit_mis.loan_master_table loanmaster  ON loanmaster.digitalLoanAccountId = r.digitalLoanAccountId
#   left join deliquency del on del.loanAccountNumber = loanmaster.loanAccountNumber
#    left join(SELECT DISTINCT mer_refferal_code, mer_name mer_name,store_type,store_tagging FROM `dl_loans_db_raw.tdbk_merchant_refferal_mtb`
#   left join worktable_datachampions.TARGET_SPLIT P on P.STORE_NAME = mer_name
#  qualify row_number() over(partition by mer_refferal_code order by  created_dt desc)=1) sil_category on loanmaster.purpleKey=sil_category.mer_refferal_code
#   where 
#   loanmaster.flagDisbursement = 1
#   and loanmaster.disbursementDateTime is not null
#   and flg_mature_fpd0 = 1
#   )
#   select *
#   from base
#   where credo_score is not null
#  qualify row_number() over(partition by digitalLoanAccountId, modelVersionId order by appln_submit_datetime) = 1
#    ;
#   """
# dfd = client.query(sq).to_dataframe()
# # dfd = dfd.drop_duplicates(keep='first')
# print(f"The shape of the dataframe downloaded is:\t {dfd.shape}")
# dfd.head()

In [ ]:
# df2 = dfd.copy()

### Concat 

In [40]:
# # df_concat = pd.concat([df1, df2], ignore_index=True)
# # print(f"The shape of the concatenated dataframe is:\t {df_concat.shape}")

# # df_concat = (df_concat
# #              .sort_values(by=['digitalLoanAccountId', 'Data_selection'],
# #                           key=lambda s: s.map({'Train': 0, 'Test': 1}))
# #              .drop_duplicates(subset=['digitalLoanAccountId'], keep='first')
# #              .reset_index(drop=True))
# # print(f"The shape of the concatenated dataframe is:\t {df_concat.shape}")
# # df_concat.head()


# # 1) Get all IDs present in Train
# train_ids = set(df2['digitalLoanAccountId'])

# # 2) Keep only Test rows whose ID is NOT in Train
# df1_no_dupes = df1[~df1['digitalLoanAccountId'].isin(train_ids)]

# # 3) Concatenate
# df_concat = pd.concat([df1_no_dupes, df2], ignore_index=True)

# print(f"The shape of the concatenated dataframe is:\t {df_concat.shape}")
# df_concat.head()

df_concat = df1.copy()


In [41]:
# # df_concat = df1.copy()

df_concat['gamma_stack_model_cash_score'] = pd.to_numeric(df_concat['gamma_stack_model_cash_score'], errors='coerce')

In [42]:
df_concat['risk_segment'] = df_concat['risk_segment'].fillna('NA')
df_concat['risk_segment_final'] = df_concat['risk_segment_final'].fillna('NA')

In [43]:
fact_table, dimension_table = calculate_periodic_gini_prod_ver_trench_dimfact(
    df_concat, 
    'gamma_stack_model_cash_score', 
    'deffspd30', 
    'FSPD30',
    data_selection_column='Data_selection',
    model_version_column='modelVersionId',
    trench_column='trenchCategory',
    loan_type_column='new_loan_type',
    loan_product_type_column='loan_product_type',
    ostype_column='osType',  
    risk_segment_column='risk_segment',
    risk_segment_final_column='risk_segment_final',
    account_id_column='digitalLoanAccountId'
)

TypeError: DataFrame.reset_index() got an unexpected keyword argument 'name'

In [45]:
if len(df_concat) > 0:
    fact_table, dimension_table = update_tables(fact_table, dimension_table, model_name='gamma_stack_model_cash', product='CASH')
    print(f"The shape of the fact table is:\t {fact_table.shape}")
    print(f"The shape of the dimension table is:\t {dimension_table.shape}")

In [ ]:
# Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.fact_table3"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(fact_table, facttable_id, job_config=job_config)
job.result()  # Wait for the job to complete

C:\Users\Dwaipayan\AppData\Roaming\Python\Python312\site-packages\google\cloud\bigquery\_pandas_helpers.py:483: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=484fbd75-e8c9-4ae6-98e8-a776f955e588>

In [ ]:
# Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.dimension_table3"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dimension_table, dimtable_id, job_config=job_config)
job.result()  # Wait for the job to complete

C:\Users\Dwaipayan\AppData\Roaming\Python\Python312\site-packages\google\cloud\bigquery\_pandas_helpers.py:483: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=d9f4b2e0-c7b0-4788-9dfa-842b0134357e>

### FSTPD30 

### Test 

In [ ]:

sq = """
with b1 as 
(select * from prj-prod-dataplatform.audit_balance.ml_model_run_details 
where modelDisplayName like 'gamma_stack_model_cash' 
qualify row_number() over(partition by customerId order by publish_time desc) = 1
),
b2 as 
(select b1.customerId, lmt.digitalLoanAccountId, lmt.disbursementDateTime, b1.crifApplicationId 
, b1.prediction, b1.errorDetail,
b1.deployedModelId,
b1.model,
b1.modelDisplayName,
b1.modelVersionId,
b1.calcFeature,
b1.start_time,
b1.end_time,
b1.subscription_name,
b1.message_id,
b1.publish_time,
b1.attributes,
b1.trenchCategory,
b1.deviceOs
from b1
left join `risk_credit_mis.loan_master_table` lmt
on lmt.customerId = cast(b1.customerId as numeric)
and date(lmt.disbursementDateTime) > date(b1.publish_time)
where lmt.disbursementDateTime is not null
),
modelname as 
(select 
mmrd.customerId,mmrd.digitalLoanAccountId,prediction,start_time,end_time,modelDisplayName,modelVersionId, 
    case when trenchCategory is null then (case when mt.ln_user_type='1_Repeat Applicant' then 'Trench 3'
    when mt.ln_user_type <>'1_Repeat Applicant' and DATE_DIFF(current_date(), mt.onb_tsa_onboarding_datetime, DAY) >30 then 'Trench 2'
    else 'Trench1' end) 
     when trenchCategory = '' then (case when mt.ln_user_type='1_Repeat Applicant' then 'Trench 3'
    when mt.ln_user_type <>'1_Repeat Applicant' and DATE_DIFF(current_date(), mt.onb_tsa_onboarding_datetime, DAY) >30 then 'Trench 2'
    else 'Trench 1' end) 
    else trenchCategory end  as trenchCategory,
    REPLACE(REPLACE(calcFeature, "'", '"'), "None", "null") AS calcFeature,
    deviceOs osType,                                                                                                                --- Added ostype
  from b2 mmrd
  left join prj-prod-dataplatform.risk_credit_mis.model_loan_score_mart mt on mt.digitalLoanAccountId = mmrd.digitalLoanAccountId
)
,
deliquency as
(select loanAccountNumber,
case when obs_min_inst_def0 >= 1 and min_inst_def0 = 1 then 1 else 0 end deffpd0,
case when obs_min_inst_def10 >=1 and min_inst_def10 =1 then 1 else 0 end deffpd10,
case when obs_min_inst_def30 >=1 and min_inst_def30 =1 then 1 else 0 end deffpd30,
case when obs_min_inst_def30 >=2 and min_inst_def30 in (1,2) then 1 else 0 end deffspd30,
case when obs_min_inst_def30 >=3 and min_inst_def30 in (1,2,3) then 1 else 0 end deffstpd30,
case when obs_min_inst_def0 >= 1 then 1 else 0 end flg_mature_fpd0,
case when obs_min_inst_def10 >=1 then 1 else 0 end flg_mature_fpd10,
case when obs_min_inst_def30 >=1 then 1 else 0 end flg_mature_fpd30,
case when obs_min_inst_def30 >=2 then 1 else 0 end flg_mature_fspd_30,
case when obs_min_inst_def30 >=3 then 1 else 0 end flg_mature_fstpd_30
from prj-prod-dataplatform.risk_credit_mis.loan_deliquency_data),
base as 
  (select distinct r.customerId,
  r.digitalLoanAccountId,
  loanmaster.loanAccountNumber,
  r.modelDisplayName,
  r.prediction gamma_stack_model_cash_score,
  coalesce(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime),  cast(r.start_time as datetime)) AS appln_submit_datetime,
  date(loanmaster.disbursementDateTime) disbursementdate,
  format_date('%Y-%m', coalesce(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime),  cast(r.start_time as datetime))) as Application_month, 
  'Prod' Data_selection,  
  del.deffstpd30,
  del.flg_mature_fstpd_30,
  loanmaster.new_loan_type,
  modelVersionId, trenchCategory,
    case when loanmaster.loantype='BNPL' and store_type =1 then 'Appliance'
    when loanmaster.loantype='BNPL' and store_type =2 then 'Mobile' 
    when loanmaster.loantype='BNPL' and store_type =3 then 'Mall' 
    when loanmaster.loantype='BNPL' and store_type not in (1,2,3) then store_tagging
    else 'not applicable' end as loan_product_type,
    coalesce((case when lower(r.osType) like '%andro%' then 'android'
                  when lower(r.osType) like '%os%' then 'ios' else lower(r.osType) end), 
            (case when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%andro%' then 'android'
                  when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%os%' then 'ios'
                  when lower(loanmaster.deviceType) like '%andro%' then 'android'
                  else 'ios' end)
            ) as osType
  from modelname r
  left join risk_credit_mis.loan_master_table loanmaster  ON loanmaster.digitalLoanAccountId = r.digitalLoanAccountId
  inner join deliquency del on del.loanAccountNumber = loanmaster.loanAccountNumber
   left join(SELECT DISTINCT mer_refferal_code, mer_name mer_name,store_type,store_tagging FROM `dl_loans_db_raw.tdbk_merchant_refferal_mtb`
  left join worktable_datachampions.TARGET_SPLIT P on P.STORE_NAME = mer_name
 qualify row_number() over(partition by mer_refferal_code order by  created_dt desc)=1) sil_category on loanmaster.purpleKey=sil_category.mer_refferal_code
  where loanmaster.flagDisbursement = 1
  and loanmaster.disbursementDateTime is not null
  and r.prediction is not null
  and del.flg_mature_fstpd_30 = 1
  ),
  segmentdata AS (
SELECT loan.customerid , loan.digitalLoanAccountId, trench_category.trenchCategory,
loan.offer_id,
CASE
WHEN COALESCE(trench1_seg.risk_segment) IS NULL
THEN 'Unsegmented'
ELSE COALESCE(trench1_seg.risk_segment)
END AS risk_segment,
appVersion,flagApproval,tsa_onboarding_time,
IF(applicationStatus  in ('COMPLETED','ACTIVATED','APPROVED') , 'Loan Approved', 'Loan Not Approved') AS loan_application_status,
--if(disbursementDateTime is not null, 'Loan Disbursed', 'Loan Not Approved') loan_application_status
DATE(decision_date) AS application_date  
from
(select distinct digitalLoanAccountId, customerId,applicationStatus,disbursementDateTime,date(decision_date) decision_date, appVersion,flagApproval,tsa_onboarding_time,offer_id from `risk_credit_mis.loan_master_table`
  where date(decision_date) >=date('2025-11-10') and new_loan_type='Quick'
--QUALIFY ROW_NUMBER() OVER(PARTITION BY customerId ORDER BY decision_date desc)=1
) loan
left join (SELECT digitalLoanAccountId,case when trenchCategory='Trench 1' then 'Trench-1'  when trenchCategory='Trench 2' then 'Trench-2'
when trenchCategory='Trench 3' then 'Trench-3' end as trenchCategory
,publish_time FROM `audit_balance.ml_model_run_details`
where modelDisplayName='gamma_stack_model_cash'
qualify row_number() over(partition by digitalLoanAccountId order by publish_time desc)=1
) trench_category on trench_category.digitalLoanAccountId=loan.digitalLoanAccountId
LEFT JOIN (
SELECT
cust_id,risk_segment,created_date,created_by,offer_id FROM `dl_loans_db_raw.tdbk_loan_offers_trx`
WHERE offer_type='SEGMENTED_ACL'
--AND created_by='GCP-API-CALL'
--QUALIFY ROW_NUMBER() OVER(PARTITION BY cust_id ORDER BY created_date desc)=1
) trench1_seg ON trench1_seg.offer_id=loan.offer_id

),
b3 as 
(select 
  b.customerId,  
  b.digitalLoanAccountId,
  b.loanAccountNumber,
  b.modelDisplayName,
  b.gamma_stack_model_cash_score,
  b.appln_submit_datetime,
  b.disbursementdate,
  b.Application_month,
  b.Data_selection,
  b.deffstpd30,
  b.flg_mature_fstpd_30 flg_mature_ftspd30,
  b.new_loan_type,
  b.modelVersionId,
  b.trenchCategory,
  b.loan_product_type,
  b.osType,
  sd.offer_id,
  sd.risk_segment,
  frs.risk_segment_final
from base b
left join segmentdata sd on sd.digitalLoanAccountId = b.digitalLoanAccountId
LEFT JOIN (
  select digitalLoanAccountid, risk_segment_final from prj-prod-dataplatform.dl_loans_db_raw.tdbk_loan_poi3_response where risk_segment_final is not null
qualify row_number() over(partition by digitalLoanAccountid order by created_dt desc) = 1
          ) frs on frs.digitalLoanAccountId = b.digitalLoanAccountId
qualify row_number() over(partition by b.digitalLoanAccountId, b.modelVersionId order by b.appln_submit_datetime) = 1
)
select * from b3 where lower(new_loan_type) not like '%sil%'
;
"""
dfd = client.query(sq).to_dataframe()
# dfd = dfd.drop_duplicates(keep='first')
print(f"The shape of the dataframe downloaded is:\t {dfd.shape}")
dfd.head()



The shape of the dataframe downloaded is:	 (0, 19)


,customerId,digitalLoanAccountId,loanAccountNumber,modelDisplayName,gamma_stack_model_cash_score,appln_submit_datetime,disbursementdate,Application_month,Data_selection,deffstpd30,flg_mature_fspd30,new_loan_type,modelVersionId,trenchCategory,loan_product_type,osType,offer_id,risk_segment,risk_segment_final


In [47]:
df1 = dfd.copy()

### Train 

In [48]:
# sq = """
# with modelname as 
#   (  SELECT
#     mmrd.customerId,mmrd.digitalLoanAccountId,prediction,start_time,end_time,modelDisplayName,modelVersionId, 
#   case when trenchCategory is null then (case when mt.ln_user_type='1_Repeat Applicant' then 'Trench 3'
#     when mt.ln_user_type <>'1_Repeat Applicant' and DATE_DIFF(current_date(), mt.onb_tsa_onboarding_datetime, DAY) >30 then 'Trench 2'
#     else 'Trench1' end) 
#      when trenchCategory = '' then (case when mt.ln_user_type='1_Repeat Applicant' then 'Trench 3'
#     when mt.ln_user_type <>'1_Repeat Applicant' and DATE_DIFF(current_date(), mt.onb_tsa_onboarding_datetime, DAY) >30 then 'Trench 2'
#     else 'Trench 1' end) 
#     else trenchCategory end  as trenchCategory,
#     REPLACE(REPLACE(calcFeature, "'", '"'), "None", "null") AS calcFeature,
#     Data_selection, deviceOs osType,
#     FROM prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116 mmrd
#   left join prj-prod-dataplatform.risk_credit_mis.model_loan_score_mart mt on mt.digitalLoanAccountId = mmrd.digitalLoanAccountId
#   WHERE modelDisplayName in ('Alpha-Cash-Stack-Model', 'alpha_stack_model_cash')
#   -- and modelVersionId = 'v1'
#       ),
#   deliquency as
# (select loanAccountNumber,
# case when obs_min_inst_def0 >= 1 and min_inst_def0 = 1 then 1 else 0 end deffpd0,
# case when obs_min_inst_def10 >=1 and min_inst_def10 =1 then 1 else 0 end deffpd10,
# case when obs_min_inst_def30 >=1 and min_inst_def30 =1 then 1 else 0 end deffpd30,
# case when obs_min_inst_def30 >=2 and min_inst_def30 in (1,2) then 1 else 0 end deffspd30,
# case when obs_min_inst_def30 >=3 and min_inst_def30 in (1,2,3) then 1 else 0 end deffstpd30,
# case when obs_min_inst_def0 >= 1 then 1 else 0 end flg_mature_fpd0,
# case when obs_min_inst_def10 >=1 then 1 else 0 end flg_mature_fpd10,
# case when obs_min_inst_def30 >=1 then 1 else 0 end flg_mature_fpd30,
# case when obs_min_inst_def30 >=2 then 1 else 0 end flg_mature_fspd_30,
# case when obs_min_inst_def30 >=3 then 1 else 0 end flg_mature_fstpd_30
# from prj-prod-dataplatform.risk_credit_mis.loan_deliquency_data),
# base as 
#   (select distinct r.customerId,
#   r.digitalLoanAccountId,
#   loanmaster.loanAccountNumber,
#   r.modelDisplayName,
#   coalesce(cast(JSON_VALUE(SAFE.PARSE_JSON(CAST(calcFeature AS STRING)), '$.credo_score')AS FLOAT64), 
#   cast(JSON_VALUE(SAFE.PARSE_JSON(CAST(calcFeature AS STRING)), '$.c_credo_score')AS FLOAT64)) AS credo_score,
#   calcFeature,
#   coalesce(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime),  cast(r.start_time as datetime)) AS appln_submit_datetime,
#   date(loanmaster.disbursementDateTime) disbursementdate,
#   format_date('%Y-%m', coalesce(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime),  cast(r.start_time as datetime))) as Application_month, 
#   Data_selection,  
#   deffpd0,
#   flg_mature_fpd0,
#   loanmaster.new_loan_type,
#   modelVersionId, trenchCategory,
#     case when loanmaster.loantype='BNPL' and store_type =1 then 'Appliance'
#     when loanmaster.loantype='BNPL' and store_type =2 then 'Mobile' 
#     when loanmaster.loantype='BNPL' and store_type =3 then 'Mall' 
#     when loanmaster.loantype='BNPL' and store_type not in (1,2,3) then store_tagging
#     else 'not applicable' end as loan_product_type,
#        coalesce((case when lower(r.osType) like '%andro%' then 'android'
#                   when lower(r.osType) like '%os%' then 'ios' else lower(r.osType) end), 
#             (case when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%andro%' then 'android'
#                   when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%os%' then 'ios'
#                   when lower(loanmaster.deviceType) like '%andro%' then 'android'
#                   else 'ios' end)
#             ) as osType
#   from modelname r
#   left join risk_credit_mis.loan_master_table loanmaster  ON loanmaster.digitalLoanAccountId = r.digitalLoanAccountId
#   left join deliquency del on del.loanAccountNumber = loanmaster.loanAccountNumber
#    left join(SELECT DISTINCT mer_refferal_code, mer_name mer_name,store_type,store_tagging FROM `dl_loans_db_raw.tdbk_merchant_refferal_mtb`
#   left join worktable_datachampions.TARGET_SPLIT P on P.STORE_NAME = mer_name
#  qualify row_number() over(partition by mer_refferal_code order by  created_dt desc)=1) sil_category on loanmaster.purpleKey=sil_category.mer_refferal_code
#   where 
#   loanmaster.flagDisbursement = 1
#   and loanmaster.disbursementDateTime is not null
#   and flg_mature_fpd0 = 1
#   )
#   select *
#   from base
#   where credo_score is not null
#  qualify row_number() over(partition by digitalLoanAccountId, modelVersionId order by appln_submit_datetime) = 1
#    ;
#   """
# dfd = client.query(sq).to_dataframe()
# # dfd = dfd.drop_duplicates(keep='first')
# print(f"The shape of the dataframe downloaded is:\t {dfd.shape}")
# dfd.head()

In [49]:
# df2 = dfd.copy()

### Concat 

In [50]:
# # df_concat = pd.concat([df1, df2], ignore_index=True)
# # print(f"The shape of the concatenated dataframe is:\t {df_concat.shape}")

# # df_concat = (df_concat
# #              .sort_values(by=['digitalLoanAccountId', 'Data_selection'],
# #                           key=lambda s: s.map({'Train': 0, 'Test': 1}))
# #              .drop_duplicates(subset=['digitalLoanAccountId'], keep='first')
# #              .reset_index(drop=True))
# # print(f"The shape of the concatenated dataframe is:\t {df_concat.shape}")
# # df_concat.head()


# # 1) Get all IDs present in Train
# train_ids = set(df2['digitalLoanAccountId'])

# # 2) Keep only Test rows whose ID is NOT in Train
# df1_no_dupes = df1[~df1['digitalLoanAccountId'].isin(train_ids)]

# # 3) Concatenate
# df_concat = pd.concat([df1_no_dupes, df2], ignore_index=True)

# print(f"The shape of the concatenated dataframe is:\t {df_concat.shape}")
# df_concat.head()

df_concat = df1.copy()


In [51]:
# # df_concat = df1.copy()

df_concat['gamma_stack_model_cash_score'] = pd.to_numeric(df_concat['gamma_stack_model_cash_score'], errors='coerce')

In [52]:
df_concat['risk_segment'] = df_concat['risk_segment'].fillna('NA')
df_concat['risk_segment_final'] = df_concat['risk_segment_final'].fillna('NA')

In [53]:
fact_table, dimension_table = calculate_periodic_gini_prod_ver_trench_dimfact(
    df_concat, 
    'gamma_stack_model_cash_score', 
    'deffspd30', 
    'FSPD30',
    data_selection_column='Data_selection',
    model_version_column='modelVersionId',
    trench_column='trenchCategory',
    loan_type_column='new_loan_type',
    loan_product_type_column='loan_product_type',
    ostype_column='osType',  
    risk_segment_column='risk_segment',
    risk_segment_final_column='risk_segment_final',
    account_id_column='digitalLoanAccountId'
)

ValueError: Missing required columns. Need: ['disbursementdate', 'gamma_stack_model_cash_score', 'deffspd30']

In [54]:
if len(df_concat) > 0:
    fact_table, dimension_table = update_tables(fact_table, dimension_table, model_name='gamma_stack_model_cash', product='CASH')
    print(f"The shape of the fact table is:\t {fact_table.shape}")
    print(f"The shape of the dimension table is:\t {dimension_table.shape}")

In [ ]:
# Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.fact_table3"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(fact_table, facttable_id, job_config=job_config)
job.result()  # Wait for the job to complete

C:\Users\Dwaipayan\AppData\Roaming\Python\Python312\site-packages\google\cloud\bigquery\_pandas_helpers.py:483: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=484fbd75-e8c9-4ae6-98e8-a776f955e588>

In [ ]:
# Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.dimension_table3"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dimension_table, dimtable_id, job_config=job_config)
job.result()  # Wait for the job to complete

C:\Users\Dwaipayan\AppData\Roaming\Python\Python312\site-packages\google\cloud\bigquery\_pandas_helpers.py:483: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=d9f4b2e0-c7b0-4788-9dfa-842b0134357e>

## Trench 2

### FPD0


#### Test